# Python for Data Engineering

*Converted from `content/python.md`.* Every `##` section of the source document is a markdown cell and every Python code block is a code cell. Snippets in SQL, PySpark, shell, YAML, Scala and other languages stay as formatted blocks because they need a specific engine to run, and the code is illustrative: check library and version requirements before running it.

Python is the default glue language of data engineering: orchestrators are written in it, engines expose a first-class API, and most pipeline code is Python even when the compute runs in Spark or a warehouse. The interview bar is not algorithmic trivia; it is whether you can write pipeline code that is memory-safe, idempotent, observable, testable, and reproducibly packaged.

Three themes run through this page: **scale** (processing more than memory allows), **correctness** (idempotent writes, retries, dead-letter handling), and **operability** (structured logs, pinned environments, CI tests).

## Why is Python widely used in data engineering?

Python is popular because it is productive, readable, and has a strong ecosystem for data processing, orchestration, and integration. It is commonly used for glue code, transformations, APIs, and automation, even when the heavy compute runs on distributed engines.

What that looks like in a real stack: **orchestration** (Airflow, Dagster), **transformation** (PySpark, Polars, pandas, SQLAlchemy), **ingestion** (`httpx`, `boto3`, Kafka clients), and **testing/CI** (pytest, Great Expectations). The trade-off to name: Python is slow per CPU-cycle and heavy per process, so the winning pattern is "Python for control flow and small data, native or distributed engines for heavy lifting."

In [ ]:
# Python usually plays the control-plane role: orchestrate, do not crunch
from airflow.sdk import dag, task


@dag(dag_id="orders_elt", schedule="0 3 * * *", start_date=None, tags=["elt"])
def orders_elt():
    @task
    def extract() -> str:
        return extract_to_s3("orders", "2026-10-04")

    @task
    def transform(path: str) -> int:
        return run_spark_job("jobs/transform_orders.py", path)

    transform(extract())


orders_elt()

## How do iterators and generators help with large data processing?

Iterators and generators allow streaming data instead of loading everything into memory. They enable processing large datasets row-by-row or batch-by-batch. This reduces memory usage and often improves stability of ETL jobs.

Gotchas: a generator is **single-use**; `yield` defers side effects to consumption; `itertools.islice`/`batched` compose well; and batching avoids per-row overhead.

In [ ]:
# Streaming pipeline: parse -> batch -> write with memory bounded by batch size
import csv
import itertools
from collections.abc import Iterable, Iterator
from typing import Any


def read_rows(path: str) -> Iterator[dict[str, str]]:
    with open(path, newline="", encoding="utf-8") as handle:
        yield from csv.DictReader(handle)


def batched(items: Iterable[Any], size: int) -> Iterator[list[Any]]:
    iterator = iter(items)
    while chunk := list(itertools.islice(iterator, size)):
        yield chunk


def run(path: str, batch_size: int = 10_000) -> int:
    return sum(write_batch(batch) for batch in batched(read_rows(path), batch_size))

Each stage is independently testable with a list, and peak memory tracks `batch_size` rather than the file.

## What is the difference between threads, multiprocessing, and async IO in Python?

Threads are useful for IO-bound workloads (network, disk) but limited by the GIL for CPU-bound tasks. Multiprocessing runs multiple processes and can use multiple CPU cores for CPU-bound work. Async IO uses an event loop to handle many concurrent IO operations efficiently when tasks spend time waiting.

In [ ]:
# Async IO: thousands of concurrent requests without thousands of threads
import asyncio

import httpx


async def fetch(client: httpx.AsyncClient, url: str) -> dict:
    response = await client.get(url, timeout=30)
    response.raise_for_status()
    return response.json()


async def fetch_many(urls: list[str], concurrency: int = 100) -> list[dict]:
    limits = httpx.Limits(max_connections=concurrency)
    async with httpx.AsyncClient(limits=limits) as client:
        return await asyncio.gather(*(fetch(client, url) for url in urls))

For CPU-bound work swap in `ProcessPoolExecutor`; for blocking IO that is not async-friendly, a `ThreadPoolExecutor` with bounded workers is often the simplest correct choice. Measure whether the workload is IO-bound or CPU-bound first.

## What is the GIL and why does it matter?

The Global Interpreter Lock (GIL) allows only one thread to execute Python bytecode at a time in CPython. This limits CPU-bound parallelism with threads. Many data engineering tasks are IO-bound, so threads can still help, but CPU-bound tasks often require multiprocessing or native libraries.

Recent nuance: CPython 3.13 introduced an **experimental free-threaded build** (PEP 703) where the GIL can be disabled, though it is not the default; the GIL is **released** around blocking IO and inside many C extensions (NumPy, Arrow, compression); and it is a CPython implementation detail, not a language rule.

In [ ]:
# If 4 threads take ~4x one thread, the GIL is serializing pure-Python CPU work
import time
from concurrent.futures import ThreadPoolExecutor


def cpu_task(n: int) -> int:
    return sum(i * i for i in range(n))


start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    list(pool.map(cpu_task, [2_000_000] * 4))
print(f"4 threads took {time.perf_counter() - start:.2f}s")

When the hot path sits in a GIL-releasing native library, threading scales much better; once it does not, move the work to processes or a distributed engine.

## How do you read and write Parquet efficiently in Python?

Using `pyarrow` is common because it supports efficient columnar reads, predicate pushdown (engine-dependent), and schema handling. You should select only needed columns, write reasonably sized row groups/files, and avoid creating many tiny files. For large datasets, integrate with a distributed engine rather than pure local processing.

In [ ]:
# Column projection + pushed-down filter, then a typed write
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq

dataset = ds.dataset("s3://acme-curated/orders/", format="parquet", partitioning=["order_date"])
table = dataset.to_table(
    columns=["order_id", "customer_id", "amount"],
    filter=(pc.field("order_date") == "2026-10-04") & (pc.field("amount") > 0),
)

schema = pa.schema([("order_id", pa.string()), ("customer_id", pa.string()),
                    ("amount", pa.decimal128(12, 2))])
with pq.ParquetWriter("orders/ds=2026-10-04/part-00000.parquet", schema,
                      compression="zstd", use_dictionary=True) as writer:
    for batch in table.select(schema.names).to_batches(max_chunksize=100_000):
        writer.write_batch(batch)

Rules: never write one file per row from a local process; size files at 128-512 MB; use `zstd` cold and `snappy` hot; read with an explicit schema in production; use `pyarrow.fs`/`fsspec` for remote paths; and hand off to Spark/Polars when the work needs shuffles or hundreds of GB.

## How do you process large CSV files without running out of memory?

You can stream the file in chunks (for example using chunked readers), process line-by-line, or use a library that supports streaming and type inference control. You also avoid loading wide columns you do not need, and you write results incrementally. For very large data, consider Parquet and/or distributed processing.

In [ ]:
# pandas chunks: aggregate incrementally, never hold the whole file
import pandas as pd

agg: dict[str, float] = {}
for chunk in pd.read_csv(
    "orders_2026.csv",
    usecols=["order_id", "amount", "country"],
    dtype={"amount": "float64", "country": "category"},
    chunksize=1_000_000,
):
    for country, total in chunk.groupby("country", observed=True)["amount"].sum().items():
        agg[country] = agg.get(country, 0.0) + total

```bash
# One-off scale: SQL over a CSV larger than memory (moves parsing into DuckDB)
duckdb -c "COPY (SELECT country, sum(amount) FROM read_csv_auto('orders_2026.csv') GROUP BY country) TO 'by_country.parquet' (FORMAT PARQUET);"
```

The theme is to aggregate or filter incrementally and emit as you go, so peak memory depends on chunk size and aggregation state rather than file size. Set explicit dtypes, because inference is itself a memory and time cost.

## How do you implement retries with exponential backoff?

Retries should include exponential backoff and jitter to avoid thundering herds. You should only retry on transient errors (timeouts, rate limits) and limit total retry time. Always make operations idempotent or use deduplication to prevent duplicates on retries.

In [ ]:
# Capped exponential backoff with full jitter, or use tenacity's
# wait_exponential_jitter + retry_if_exception_type for the declarative version
import random
import time
from collections.abc import Callable
from typing import TypeVar

T = TypeVar("T")


class TransientError(Exception):
    """Raised for retryable failures (timeouts, 429, 503)."""


def retry(fn: Callable[[], T], *, attempts: int = 5, base: float = 0.5, cap: float = 30.0) -> T:
    last: Exception | None = None
    for attempt in range(1, attempts + 1):
        try:
            return fn()
        except TransientError as exc:
            last = exc
            if attempt == attempts:
                break
            time.sleep(random.uniform(0, min(cap, base * 2 ** (attempt - 1))))
    raise RuntimeError(f"retries exhausted after {attempts} attempts") from last

Rules: retry only transient classes; honor `Retry-After` on 429/503; cap total time, not just attempts; use idempotency keys on writes; add jitter; and log every attempt with the attempt number.

## What logging practices are important for data pipelines?

Logs should include pipeline identifiers, run ids, input/output counts, and key timestamps. Structured logging (JSON) helps search and correlation. You should log errors with context and avoid logging sensitive data.

In [ ]:
# JSON logging with run-scoped context fields
import json
import logging
from typing import Any

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("orders")


class JsonFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        payload: dict[str, Any] = {"level": record.levelname,
                                   "logger": record.name,
                                   "message": record.getMessage()}
        payload.update(getattr(record, "extra_fields", {}))
        if record.exc_info:
            payload["traceback"] = self.formatException(record.exc_info)
        return json.dumps(payload)


handler = logging.StreamHandler()
handler.setFormatter(JsonFormatter())
log.handlers = [handler]
log.info("stage_complete", extra={"extra_fields": {"run_id": "r-123", "rows_in": 1000, "rows_out": 998}})

Standards that matter: one start/end line per stage with `run_id`, dataset, and partition; counts in and out; `logging` with a JSON formatter rather than `print`; never log secrets or raw PII; `exc_info=True` for exceptions; and a correlation id so logs join to traces and the control table.

## How do you structure a Python project for data pipelines?

A common structure separates reusable library code from job entrypoints. You keep configuration separate, use dependency injection for external services, and standardize logging and error handling. Small CLI entrypoints improve local runs and CI tests.

A common layout is `src/<pkg>/` with `config.py`, `io/`, `transforms/` (pure functions), and `jobs/` (thin CLI entrypoints), plus `tests/unit`, `tests/integration`, and `dags/` where Airflow files import the package.

Principles: business logic lives in importable modules while `jobs/` is a thin shell; config is parsed into a typed object once (`Settings.from_env()` returning a frozen dataclass); clients are injected so tests can substitute fakes; there are **no import-time side effects** (DAG parsing depends on this); and the package is installed (`uv sync`) so jobs and tests import the same code.

## How do you manage dependencies and reproducible environments?

Use pinned dependencies and a lockfile (or equivalent) and isolate environments (virtualenv/containers). Reproducibility also requires pinning Python versions and building deterministic images. CI should use the same environment definition as production.

```bash
# Reproducible workflow: lock once, install exactly what CI and prod use
uv lock                         # writes uv.lock with hashes
uv sync --frozen                # install exactly the lockfile
uv run pytest                   # tests run in the pinned environment
```

```yaml
# pyproject.toml (excerpt, shown as YAML for readability)
project:
  name: acme-data
  requires-python: ">=3.12"
  dependencies: ["pyarrow>=17,<18", "polars>=1.0,<2", "httpx>=0.27,<1"]
  optional-dependencies:
    spark: ["pyspark==3.5.1"]
    dev: ["pytest>=8", "ruff", "mypy"]
```

Anti-patterns: unpinned transitive dependencies (behavior changes silently on rebuild); installing packages at task runtime; different environments for CI and production; and hand-maintaining hundreds of `==` pins instead of using a lockfile tool. Container images should pin the base Python by digest, `COPY uv.lock`, run `uv sync --frozen`, and then copy `src` - the same lock the tests use.

## How do you test data transformations in Python?

Use small fixture datasets and assert expected outputs (golden tests). Add property tests for invariants (non-negative amounts, uniqueness). Integration tests validate real connectors and schemas, but they should be isolated and stable.

Keep transformations **pure** so the fast unit layer covers most logic, then add one integration test per connector behind a marker.

In [ ]:
# tests/unit/test_orders.py
import pandas as pd
import pytest

from acme_data.transforms.orders import add_tax


@pytest.fixture
def orders() -> pd.DataFrame:
    return pd.DataFrame({"order_id": ["o1", "o2"], "amount_cents": [1000, 2000],
                         "country": ["DE", "US"]})


def test_add_tax(orders):
    out = add_tax(orders, rates={"DE": 0.19, "US": 0.0})
    assert out["amount_cents"].tolist() == [1190, 2000]


# Property test: for any list of non-negative amounts the tax never decreases them
# @given(st.lists(st.integers(min_value=0), min_size=1))
# def test_tax_never_decreases(amounts): ...

```bash
uv run pytest tests/unit -q                       # always, in CI
uv run pytest -m integration --maxfail=1          # opt-in, needs a test warehouse
```

Keep fixtures small enough to verify by eye, and assert business rules rather than snapshotting current output - change-detector tests pass forever and catch nothing.

## How do you profile and optimize slow Python code?

Profile first to find hotspots (CPU, IO, memory). Common fixes include vectorizing operations, reducing data copies, using efficient data structures, and offloading heavy compute to native libraries or distributed engines. Optimization should be guided by measurable impact.

Pick the tool by question: `cProfile` + `pstats` for wall-clock hotspots, `line_profiler` for a specific line, `tracemalloc`/`memray` for peak or leaking memory, and `py-spy dump` to see whether a running process is IO- or CPU-bound.

In [ ]:
# cProfile + pstats: sort by cumulative time to find the real hotspot
import cProfile
import pstats


def main() -> None:
    run_pipeline("2026-10-04")


profiler = cProfile.Profile()
profiler.enable()
main()
profiler.disable()
pstats.Stats(profiler).sort_stats("cumulative").print_stats(20)

Common wins in order of impact: vectorize row-wise `.apply`/loops (`(df["amount"] * (1 + df["tax_rate"])).sum()` instead of `iterrows`); avoid copies (`copy=False`, reuse buffers, prefer Arrow over `to_list()`); choose the right container; reduce IO by batching and streaming; and move compute to SQL/Spark when the data never needed to be in Python.

## How do you write memory-safe Python ETL code?

Memory-safe ETL code avoids loading unbounded data into lists or DataFrames. Prefer iterators, chunked reads, streaming writes, and explicit column selection. This is especially important for scripts that run in containers with fixed memory limits.

Example:

In [ ]:
import csv

def active_users(path):
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["status"] == "active":
                yield {
                    "user_id": int(row["user_id"]),
                    "email": row["email"].lower(),
                }

for user in active_users("users.csv"):
    send_to_api(user)

This processes one row at a time instead of building a full in-memory collection. In interviews, call out the trade-off: row-by-row code is memory efficient, while vectorized/batch processing can be faster when data fits in memory.

Checklist: bound every buffer (chunk, batch, and queue sizes); select columns in the reader rather than dropping later; use `category` dtype for low-cardinality strings and downcast numerics; never `concat` inside a loop (collect and concat once, or stream to disk); and cap concurrency so parallel fetches cannot exceed the container limit. Test with a larger-than-expected input, because a pipeline that only works on the sample partition is an outage waiting to happen.

## How do you design Python code so a data pipeline is restartable?

A restartable pipeline can safely resume after failure without corrupting downstream data. Use idempotent writes, deterministic output paths, checkpoints/control tables, and stable run identifiers. Avoid hidden global state and make external side effects explicit.

Example pattern:

In [ ]:
def run_partition(ds: str) -> None:
    source = f"s3://raw/events/ds={ds}/"
    staging = f"s3://warehouse/events_staging/run_ds={ds}/"
    final = f"s3://warehouse/events/ds={ds}/"

    records = transform(read_json(source))
    write_parquet(records, staging)
    validate_partition(staging)
    replace_partition(staging, final)
    mark_success("events", ds)

The important idea is to write to staging first, validate, then atomically publish or replace the final partition. If the job retries, it produces the same partition instead of appending duplicates. A control table (`platform.partition_state` with dataset, partition, status, attempt, rows_written, finished_at) tracks per-partition state so the loop can skip completed work.

Rules that make restart safe: deterministic paths (same logical date, same output); atomic publish via rename, partition overwrite, or MERGE; a stable run id passed through every stage; explicit enumeration of external side effects; and no mutable module-level state that assumes one run per process.

## How do you handle bad records in Python pipelines?

Do not let one malformed row either crash an entire batch blindly or disappear silently. A strong pipeline separates valid records from rejected records, records the reason for rejection, and emits metrics so data quality issues are visible.

In [ ]:
# Dead-letter records carry enough context to replay, plus a metric
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from typing import Any


@dataclass
class DeadLetter:
    source_file: str
    line_number: int
    error_message: str
    payload: str
    detected_at: str


def parse_with_dlq(row: dict[str, str], source_file: str, line_number: int) -> dict[str, Any] | None:
    try:
        return {"order_id": row["order_id"], "amount": float(row["amount"])}
    except (KeyError, ValueError) as exc:
        dead_letter.write(asdict(DeadLetter(
            source_file=source_file, line_number=line_number, error_message=str(exc),
            payload=json.dumps(row)[:4000], detected_at=datetime.now(timezone.utc).isoformat(),
        )))
        REJECTED.labels(pipeline="orders").inc()
        return None

In production, rejected records go to a dead-letter path/table with the source file, line number, pipeline version, and error message, so teams can fix upstream data or replay rejects after correction. Enforce a reject-rate threshold so a flood of malformed records blocks rather than silently publishing. **Failure mode:** swallowing bad rows with `try/except: pass` - the job goes green, the metric declines, and the records cannot be replayed.

## When should you use pandas, Polars, PyArrow, or Spark from Python?

Use pandas for small to medium in-memory analysis and simple transformations. Use Polars or PyArrow when you need faster columnar processing, lazy execution, or efficient Parquet/Arrow interoperability on a single machine. Use Spark/PySpark when data volume, shuffle requirements, or production architecture require distributed processing.

Example decision rule:

+ pandas: one CSV or Parquet extract that fits comfortably in memory
+ PyArrow: efficient file/schema operations, Parquet datasets, Arrow interchange
+ Polars: fast local transformations and lazy query optimization
+ PySpark: joins, aggregations, and pipelines over data too large for one machine

The interview answer should emphasize workload shape, not tool popularity. A 5 GB file may be fine in Polars on a large instance, while a 500 GB join with skewed keys usually belongs in Spark or a warehouse.

In [ ]:
# Polars: lazy scan with projection and predicate pushdown on a large local dataset
import polars as pl

result = (
    pl.scan_parquet("s3://acme-curated/orders/*/*.parquet")
    .filter(pl.col("order_date") == "2026-10-04")
    .group_by("country")
    .agg(pl.col("amount").sum().alias("revenue"))
    .collect(streaming=True)
)

Rule of thumb: **stay single-node until the data genuinely exceeds memory or the shuffle does not fit on one machine.** Polars/DuckDB push that boundary higher than pandas, and moving to Spark prematurely costs cluster operations and JVM serialization you did not need to pay.

## Real world use cases for Python for Data Engineering

### 1. E-commerce: API ingestion with retries and a dead-letter queue

A retailer pulls orders from a partner REST API that rate-limits aggressively. Roughly 4M orders/day arrive in 5-minute windows, then land in S3 and load into the warehouse.

In [ ]:
# Bounded-concurrency fetcher with retry and backoff; exhausted pages go to a DLQ
import asyncio

import httpx
from tenacity import retry, retry_if_exception_type, stop_after_attempt, wait_exponential_jitter


class RateLimited(Exception):
    pass


@retry(retry=retry_if_exception_type((RateLimited, httpx.TransportError)),
       wait=wait_exponential_jitter(initial=1, max=60), stop=stop_after_attempt(5), reraise=True)
async def fetch_page(client: httpx.AsyncClient, cursor: str | None) -> dict:
    response = await client.get("/orders", params={"cursor": cursor, "limit": 500})
    if response.status_code == 429:
        raise RateLimited(response.headers.get("Retry-After", "unknown"))
    response.raise_for_status()
    return response.json()

A page that exhausts retries is written to a quarantine prefix with its cursor, so the pull resumes from the last good cursor instead of re-reading the whole day.

**Why this design:** the API is the bottleneck, so async IO plus bounded concurrency maximizes throughput, retries with jitter absorb 429s, and a durable cursor makes a failure cost one page rather than one day.

### 2. Fintech: reconciliation of two systems with Polars

A payments team reconciles ~120M transactions/day between a core ledger and the warehouse, classifying differences before the close.

In [ ]:
# Vectorized reconciliation: a set join, not a loop over 120M rows
import polars as pl

ledger = pl.scan_parquet("s3://acme-ledger/txn/dt=2026-10-04/*.parquet").select("txn_id", "amount_cents")
warehouse = pl.scan_parquet("s3://acme-wh/txn/dt=2026-10-04/*.parquet").select("txn_id", "amount_cents")

recon = (
    ledger.join(warehouse, on="txn_id", how="full", suffix="_wh")
    .with_columns(
        pl.when(pl.col("amount_cents").is_null()).then(pl.lit("missing_in_warehouse"))
        .when(pl.col("amount_cents_wh").is_null()).then(pl.lit("missing_in_ledger"))
        .when(pl.col("amount_cents") != pl.col("amount_cents_wh")).then(pl.lit("amount_mismatch"))
        .otherwise(pl.lit("match")).alias("recon_status")
    )
    .group_by("recon_status").agg(pl.len().alias("count"))
    .collect(streaming=True)
)

**Why this design:** reconciliation is a set operation, so a vectorized join handles 120M rows on one node in minutes, and the output classifies every difference instead of returning a binary gate the finance team cannot act on.

### 3. Multi-tenant SaaS: memory-bounded export per tenant

A B2B platform lets customers download their data. Some tenants have billions of events; the export service runs in a container with a 2 GB limit and must never OOM.

In [ ]:
# Server-side cursor plus fetchmany keeps memory proportional to the batch
BATCH = 100_000


def stream_tenant_events(conn, tenant_id: str, out_path: str) -> int:
    written = 0
    with gzip.open(out_path, "wt", encoding="utf-8") as handle, conn.cursor(name="export_cursor") as cur:
        cur.itersize = BATCH
        cur.execute("SELECT event_id, event_ts FROM events WHERE tenant_id = %s", (tenant_id,))
        while rows := cur.fetchmany(BATCH):
            for row in rows:
                handle.write(json.dumps(row) + "\n")
            written += len(rows)
    return written

**Why this design:** the container limit is a hard constraint, so memory is bounded by the batch rather than the tenant, and compressed NDJSON streams bytes to disk instead of buffering them.

### 4. ML feature platform: a reproducible training snapshot

An ML team builds daily features for 40M users. Reproducibility matters because model comparisons must attribute changes to the model, not to a re-run of feature code.

In [ ]:
# Installable, versioned feature job with a content hash for the snapshot
import hashlib

import polars as pl


def build_features(input_glob: str, code_version: str) -> tuple[pl.DataFrame, str]:
    frame = (
        pl.scan_parquet(input_glob)
        .group_by("user_id")
        .agg(pl.col("event_type").eq("view").sum().alias("views_1d"),
             pl.col("amount").sum().alias("spend_1d"))
        .collect(streaming=True)
    )
    digest = hashlib.sha256(f"{code_version}{frame.shape}{frame.sum().to_dict()}".encode())
    return frame, digest.hexdigest()[:16]

**Why this design:** the job is a normal installable package with a lockfile, so CI and production run the same code and dependency versions, and a content hash identifies the exact snapshot an experiment trained on.

---

# Practice notebook: Python basics

The cells below are the exercises from `notebooks/01_python_basics.ipynb`, merged here so the topic notebook is self-contained. They are self-checking: run them top to bottom and the assertions will confirm your answers.

# Python Basics for Data Engineering Interviews

This notebook is a hands-on tour of the Python language features that come up in data-engineering interviews:
classes and methods, dunder methods, exceptions, context managers, generators, decorators, typing, and the
concurrency basics everyone gets asked about.

**How to use it**

1. Run the cells top to bottom (`Shift+Enter`).
2. Every concept has a runnable example and most have a **self-check** cell with `assert`s, so you know immediately
   whether your mental model is right.
3. The last section is a bank of interview questions with model answers.

Companion notebooks: `02_python_coding_problems.ipynb` (algorithms), `03_sql_interview_questions.ipynb` (SQL),
`04_pyspark_concepts.ipynb` (Spark), `05_interview_questions.ipynb` (mixed question bank).

## 1. Values, types and the four workhorse collections

Python is dynamically typed: names are bound to objects, and the object knows its type. The four collections you
will use constantly are `list` (ordered, mutable), `tuple` (ordered, immutable), `set` (unique, unordered) and
`dict` (key-value, insertion-ordered since 3.7).

In [1]:
numbers = [3, 1, 4, 1, 5, 9, 2, 6]
print("list:", numbers, "| sorted:", sorted(numbers), "| unique:", sorted(set(numbers)))

record = ("order-1", 42.5)                 # tuple: fixed shape
name, amount = record                      # unpacking
print("tuple:", record, name, amount)

squares = {n: n * n for n in range(5)}     # dict comprehension
evens = [n for n in numbers if n % 2 == 0] # list comprehension
print("dict:", squares)
print("evens:", evens)

text = "  Data Engineering  "
print("strip/lower/split:", text.strip().lower().split())
print("f-string:", f"{len(numbers)} values, max={max(numbers)}, total={sum(numbers)}")
print("slice:", numbers[2:5], "| step:", numbers[::2])

list: [3, 1, 4, 1, 5, 9, 2, 6] | sorted: [1, 1, 2, 3, 4, 5, 6, 9] | unique: [1, 2, 3, 4, 5, 6, 9]
tuple: ('order-1', 42.5) order-1 42.5
dict: {0: 0, 1: 1, 2: 4, 3: 9, 4: 16}
evens: [4, 2, 6]
strip/lower/split: ['data', 'engineering']
f-string: 8 values, max=9, total=31
slice: [4, 1, 5] | step: [3, 4, 5, 2]


**Exercise 1.** Write `top_n_by_value(counts, n)` that returns the `n` keys with the highest counts, breaking ties
by key name so the result is deterministic.

In [2]:
def top_n_by_value(counts: dict, n: int) -> list:
    # sort by count descending, then key ascending
    return sorted(counts, key=lambda k: (-counts[k], k))[:n]

assert top_n_by_value({"b": 2, "a": 2, "c": 5}, 2) == ["c", "a"]
assert top_n_by_value({}, 3) == []
assert top_n_by_value({"x": 1}, 5) == ["x"]
print("exercise 1 passed")

exercise 1 passed


## 2. Mutability, references and copies

Assignment binds a name to an object; it does not copy. Functions receive references, so mutating a list argument
is visible to the caller. Interviewers love this because it explains surprise bugs in pipeline code.

In [3]:
a = [1, 2, 3]
b = a                    # same object
b.append(4)
assert a == [1, 2, 3, 4] and a is b

import copy
c = a.copy()             # shallow copy
c.append(99)
assert a == [1, 2, 3, 4] and c == [1, 2, 3, 4, 99]

nested = [[1], [2]]
shallow = copy.copy(nested)
deep = copy.deepcopy(nested)
nested[0].append("boom")
assert shallow[0] == [1, "boom"]   # shallow shares the inner lists
assert deep[0] == [1]              # deep is fully independent
print("mutability checks passed")

mutability checks passed


## 3. Functions, scope and closures

Keyword arguments, `*args` / `**kwargs`, closures and the LEGB rule (Local, Enclosing, Global, Built-in) are
standard interview material.

In [4]:
def load(table, *, mode="append", **options):
    "mode is keyword-only; options collects anything else"
    return {"table": table, "mode": mode, **options}

assert load("orders", mode="overwrite", compression="zstd") == {
    "table": "orders", "mode": "overwrite", "compression": "zstd"
}

def make_counter():
    count = 0
    def increment():
        nonlocal count          # rebind the enclosing variable
        count += 1
        return count
    return increment

counter = make_counter()
assert [counter(), counter(), counter()] == [1, 2, 3]

def total(*values):             # variable positional args
    return sum(values)

assert total(1, 2, 3) == 6 and total() == 0
print("scope/closures checks passed")

scope/closures checks passed


### Pitfall: mutable default arguments and late binding

Two classics that show up in production code review questions.

In [5]:
def append_bad(item, bucket=[]):        # the default list is created once
    bucket.append(item)
    return bucket

assert append_bad("a") == ["a"]
assert append_bad("b") == ["a", "b"]    # surprising, but that is the point

def append_good(item, bucket=None):
    bucket = [] if bucket is None else bucket
    bucket.append(item)
    return bucket

assert append_good("a") == ["a"]
assert append_good("b") == ["b"]

callbacks = [lambda: i for i in range(3)]
assert [cb() for cb in callbacks] == [2, 2, 2]          # late binding
callbacks = [lambda i=i: i for i in range(3)]
assert [cb() for cb in callbacks] == [0, 1, 2]          # bound at definition
print("pitfall checks passed")

pitfall checks passed


## 4. Classes, objects and the three kinds of methods

* `__init__` initialises an instance; `self` is the instance.
* **Instance methods** take `self` and work on one object.
* **Class methods** take `cls`, often used as alternative constructors.
* **Static methods** take neither: they are namespaced helpers.

In [6]:
class Account:
    currency = "USD"                     # class attribute, shared

    def __init__(self, owner, balance=0.0):
        self.owner = owner               # instance attribute
        self.balance = balance           # per-object
        self.history = []

    def deposit(self, amount):           # instance method
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self.balance += amount
        self.history.append(("deposit", amount))
        return self.balance

    @classmethod
    def from_statement(cls, owner, amounts):    # alternative constructor
        account = cls(owner)
        for amount in amounts:
            account.deposit(amount)
        return account

    @staticmethod
    def fee_for(amount):                  # pure helper
        return round(amount * 0.01, 2)

    def __repr__(self):
        return f"Account(owner={self.owner!r}, balance={self.balance:.2f})"

account = Account.from_statement("ada", [100, 50])
assert account.balance == 150
assert Account.fee_for(200) == 2.0
assert Account.currency == "USD"
assert "balance=150.00" in repr(account)
print("class checks passed:", account)

class checks passed: Account(owner='ada', balance=150.00)


### Inheritance and polymorphism

`super()` calls the parent implementation; method resolution order (MRO) decides which class wins in a diamond.

In [7]:
class Storage:
    def __init__(self, path):
        self.path = path
    def save(self, rows):
        return f"wrote {len(rows)} rows to {self.path}"

class ParquetStorage(Storage):
    def save(self, rows):
        base = super().save(rows)                 # reuse parent behaviour
        return f"{base} as Parquet"

class DeltaStorage(Storage):
    def save(self, rows):
        return f"merged {len(rows)} rows into {self.path} (Delta)"

for backend in (ParquetStorage("s3://lake/curated"), DeltaStorage("s3://lake/delta")):
    print(backend.save([1, 2, 3]))

assert ParquetStorage("p").save([1]) == "wrote 1 rows to p as Parquet"
assert [c.__name__ for c in ParquetStorage.__mro__] == ["ParquetStorage", "Storage", "object"]

wrote 3 rows to s3://lake/curated as Parquet
merged 3 rows into s3://lake/delta (Delta)


## 5. Dunder methods: making your objects behave like built-ins

Defining `__repr__`, `__eq__`, `__hash__`, `__add__`, `__len__` and `__iter__` is the difference between a data bag
and a first-class value.

In [8]:
class Money:
    def __init__(self, cents, currency="USD"):
        self.cents = int(cents)
        self.currency = currency

    def __repr__(self):
        return f"Money({self.cents}, {self.currency!r})"

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return (self.cents, self.currency) == (other.cents, other.currency)

    def __hash__(self):
        return hash((self.cents, self.currency))     # keep eq/hash consistent

    def __add__(self, other):
        if self.currency != other.currency:
            raise ValueError("cannot add different currencies")
        return Money(self.cents + other.cents, self.currency)

    def __bool__(self):
        return self.cents != 0

price = Money(1250)
tax = Money(100)
assert price + tax == Money(1350)
assert len({Money(1), Money(1), Money(2)}) == 2     # hashable in a set
assert bool(Money(0)) is False and bool(price) is True
print("dunder checks passed:", price + tax)

dunder checks passed: Money(1350, 'USD')


## 6. Properties: validation without breaking the attribute API

`@property` turns a method into a computed attribute; the setter validates writes, so callers keep using
`obj.value = ...`.

In [9]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius          # goes through the setter

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = float(value)

    @property
    def fahrenheit(self):               # read-only computed attribute
        return self._celsius * 9 / 5 + 32

t = Temperature(25)
assert t.fahrenheit == 77.0
t.celsius = -40
assert t.fahrenheit == -40.0
try:
    Temperature(-300)
except ValueError as exc:
    assert "absolute zero" in str(exc)
print("property checks passed")

property checks passed


## 7. Dataclasses and enums

`@dataclass` generates `__init__`, `__repr__` and `__eq__`; `frozen=True` makes it hashable/immutable.
`Enum` replaces magic strings.

In [10]:
from dataclasses import dataclass, field
from enum import Enum

class Status(Enum):
    PLACED = "placed"
    SHIPPED = "shipped"
    CANCELLED = "cancelled"

@dataclass(frozen=True)
class Order:
    order_id: str
    amount: float
    status: Status = Status.PLACED
    tags: tuple = field(default_factory=tuple)

@dataclass
class Cart:
    items: list = field(default_factory=list)     # fresh list per instance

    def add(self, item):
        self.items.append(item)
        return self

order = Order("o-1", 42.5, tags=("priority",))
assert order.status is Status.PLACED
assert order == Order("o-1", 42.5, tags=("priority",))
assert len({order, Order("o-1", 42.5, tags=("priority",))}) == 1   # frozen => hashable

c1, c2 = Cart(), Cart()
c1.add("a")
assert c1.items == ["a"] and c2.items == []      # no shared default
print("dataclass/enum checks passed")

dataclass/enum checks passed


## 8. Exceptions: hierarchy, `else`, `finally`, chaining

Catch the narrowest exception you can handle, add context with `raise ... from`, and use `else`/`finally` for the
success and cleanup paths.

In [11]:
class DataQualityError(Exception):
    "Base class for pipeline data-quality failures"

class EmptyPartitionError(DataQualityError):
    def __init__(self, partition):
        super().__init__(f"partition {partition} is empty")
        self.partition = partition

def validate(rows, partition="ds=2026-10-04"):
    if rows is None:
        raise ValueError("rows is None")
    if not rows:
        raise EmptyPartitionError(partition)
    return len(rows)

steps = []
try:
    steps.append(validate([], "ds=2026-10-03"))
except EmptyPartitionError as exc:
    steps.append(f"caught {exc.partition}")
else:
    steps.append("no error")
finally:
    steps.append("cleanup")

assert steps == ["caught ds=2026-10-03", "cleanup"]
assert isinstance(EmptyPartitionError("x"), DataQualityError)

try:
    validate(None)
except ValueError as exc:
    chained = exc
assert isinstance(chained, ValueError) and "rows is None" in str(chained)
print("exception checks passed")

exception checks passed


## 9. Context managers: guaranteed setup and teardown

Any object with `__enter__` / `__exit__` works with `with`. `contextlib.contextmanager` turns a generator into one.
Typical data-engineering uses: connections, transactions, temporary files, timers.

In [12]:
from contextlib import contextmanager
import time

class Transaction:
    def __init__(self, name):
        self.name = name
        self.log = []
    def __enter__(self):
        self.log.append(f"begin {self.name}")
        return self
    def __exit__(self, exc_type, exc, tb):
        self.log.append("rollback" if exc_type else f"commit {self.name}")
        return False            # do not swallow exceptions

tx = Transaction("load-orders")
with tx:
    tx.log.append("insert rows")
assert tx.log == ["begin load-orders", "insert rows", "commit load-orders"]

failing = Transaction("bad-load")
try:
    with failing:
        raise RuntimeError("warehouse down")
except RuntimeError:
    pass
assert failing.log[-1] == "rollback"

@contextmanager
def timer():
    started = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - started
        assert elapsed >= 0

with timer():
    pass
print("context manager checks passed")

context manager checks passed


## 10. Iterators and generators: memory-safe pipelines

A generator yields values lazily, so you can process a file larger than memory. Interviewers frequently ask you to
turn a list-based function into a generator.

In [13]:
def read_batches(rows, size):
    for start in range(0, len(rows), size):
        yield rows[start:start + size]

def parse_orders(lines):
    for line in lines:
        order_id, amount = line.split(",")
        yield {"order_id": order_id, "amount": float(amount)}

def only_positive(orders):
    for order in orders:
        if order["amount"] > 0:
            yield order

raw = ["o1,10.5", "o2,-3", "o3,7"]
parsed = list(parse_orders(raw))
filtered = list(only_positive(parse_orders(raw)))
assert parsed[0] == {"order_id": "o1", "amount": 10.5}
assert [o["order_id"] for o in filtered] == ["o1", "o3"]
assert list(read_batches([1, 2, 3, 4, 5], 2)) == [[1, 2], [3, 4], [5]]

# lazy: nothing is consumed until the generator is iterated
consumed = []
def tracked():
    for i in range(3):
        consumed.append(i)
        yield i

lazy = tracked()
assert consumed == []
assert list(lazy) == [0, 1, 2] and consumed == [0, 1, 2]
print("generator checks passed")

generator checks passed


## 11. Decorators: cross-cutting behaviour

A decorator is a function that takes a function and returns a replacement. `functools.wraps` preserves metadata;
`lru_cache` memoises; retry wrappers belong in every ingestion pipeline.

In [14]:
import functools

def traced(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return fn(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

def retry(times=3):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except ConnectionError:
                    wrapper.attempts = attempt
                    if attempt == times:
                        raise
        return wrapper
    return decorator

@traced
def add(a, b):
    return a + b

assert add(1, 2) == 3 and add.calls == 1
assert add.__name__ == "add"            # functools.wraps preserved the name

state = {"fails": 2}
@retry(times=5)
def flaky():
    if state["fails"] > 0:
        state["fails"] -= 1
        raise ConnectionError("temporary")
    return "ok"

assert flaky() == "ok"

@functools.lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

assert fib(30) == 832040
assert fib.cache_info().misses <= 31
print("decorator checks passed")

decorator checks passed


## 12. Type hints

Hints document intent and let `mypy`/IDEs catch mistakes; they are not enforced at runtime. Use them on public
functions and data models.

In [15]:
from typing import Iterable, Iterator, Optional

def average(values: Iterable[float]) -> Optional[float]:
    total = 0.0
    count = 0
    for value in values:
        total += value
        count += 1
    return total / count if count else None

def running_total(values: Iterable[float]) -> Iterator[float]:
    total = 0.0
    for value in values:
        total += value
        yield total

def find_partition(partitions: dict[str, str], prefix: str) -> str | None:
    return partitions.get(prefix)

assert average([1, 2, 3]) == 2.0
assert average([]) is None
assert list(running_total([1, 2, 3])) == [1.0, 3.0, 6.0]
assert find_partition({"a": "s3://a"}, "a") == "s3://a"
print("typing checks passed")

typing checks passed


## 13. Dates, times and timezones

Store timestamps in UTC, attach a timezone, and convert only for display. `datetime.utcnow()` is deprecated; use
`datetime.now(timezone.utc)`.

In [16]:
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

started = datetime(2026, 10, 4, 12, 0, tzinfo=timezone.utc)
finished = started + timedelta(hours=2, minutes=30)
assert (finished - started).total_seconds() == 9000

as_ny = finished.astimezone(ZoneInfo("America/New_York"))
assert as_ny.hour == 10 and as_ny.utcoffset() == timedelta(hours=-4)   # EDT

parsed = datetime.fromisoformat("2026-10-04T12:00:00+00:00")
assert parsed.tzinfo is not None and parsed == started

print("utc:", started.isoformat(), "| new york:", as_ny.isoformat())

utc: 2026-10-04T12:00:00+00:00 | new york: 2026-10-04T10:30:00-04:00


## 14. Files, `pathlib` and JSON

`pathlib.Path` is the modern file API; `json` and `csv` are stdlib. The examples write to a temporary directory so
nothing is left behind.

In [17]:
import json, tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    orders = [{"order_id": "o1", "amount": 10.5}, {"order_id": "o2", "amount": 7.0}]

    ndjson = root / "orders.ndjson"
    with ndjson.open("w", encoding="utf-8") as handle:
        for order in orders:
            handle.write(json.dumps(order) + "\n")

    loaded = [json.loads(line) for line in ndjson.read_text(encoding="utf-8").splitlines()]
    assert loaded == orders
    assert ndjson.suffix == ".ndjson" and ndjson.name == "orders.ndjson"

    total = sum(order["amount"] for order in loaded)
    assert total == 17.5
    print("round-tripped", len(loaded), "orders, total", total)

round-tripped 2 orders, total 17.5


## 15. Concurrency basics: GIL, threads, processes, asyncio

* The **GIL** lets only one thread run Python bytecode at a time, so threads help **I/O-bound** work, not CPU-bound
  work.
* Use **processes** (`ProcessPoolExecutor`) to bypass the GIL for CPU-bound work.
* Use **asyncio** for many concurrent network calls in one thread.

In [18]:
from concurrent.futures import ThreadPoolExecutor
import time

def fetch_partition(partition):
    time.sleep(0.01)                    # pretend this is a network call
    return f"rows for {partition}"

partitions = [f"ds=2026-10-{day:02d}" for day in range(1, 6)]
started = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as pool:
    results = list(pool.map(fetch_partition, partitions))
elapsed = time.perf_counter() - started

assert results[0] == "rows for ds=2026-10-01" and len(results) == 5
print(f"5 mocked fetches in {elapsed*1000:.1f} ms (sequential would be ~50 ms)")

5 mocked fetches in 11.0 ms (sequential would be ~50 ms)


An `asyncio` version of the same idea (shown as code to read, not run, because a notebook already runs an event
loop):

```python
import asyncio

async def fetch(partition):
    await asyncio.sleep(0.01)
    return f"rows for {partition}"

async def main():
    return await asyncio.gather(*(fetch(p) for p in partitions))
```

Rule of thumb for interviews: threads for blocking I/O with libraries that are not async-aware; asyncio when the
whole call chain is async; processes for CPU-heavy Python; and Spark for genuinely large-scale parallelism.

## 16. Common pitfalls checklist

Mutable defaults, late binding, float equality, string concatenation in loops, shadowing builtins, and `is` vs `==`.

In [19]:
# float comparison: use math.isclose, not ==
assert 0.1 + 0.2 != 0.3
import math
assert math.isclose(0.1 + 0.2, 0.3, rel_tol=1e-9)

# build strings with join, not += in a loop
parts = [str(i) for i in range(5)]
assert "-".join(parts) == "0-1-2-3-4"

# is compares identity; == compares value
a, b = [1, 2], [1, 2]
assert a == b and a is not b

# do not shadow builtins
list_of_rows = [1, 2]        # not `list`
assert sum(list_of_rows) == 3
print("pitfall checks passed")

pitfall checks passed


## Interview questions

Answer these out loud before reading the model answers.

1. **What is the difference between a list and a tuple, and when do you prefer each?**
   Lists are mutable and used for homogeneous sequences you will grow or change; tuples are immutable, hashable and
   used for fixed-shape records (composite keys, function returns) where immutability is a feature.
2. **Explain the mutable default argument trap.**
   Default values are evaluated once at function definition, so a `def f(x, bucket=[])` shares one list across all
   calls. Use `None` and create the container inside the function.
3. **What is the difference between `__str__` and `__repr__`?**
   `__repr__` is the unambiguous developer representation (ideally `eval`-able) used in debugging; `__str__` is the
   human-friendly display used by `print`/f-strings. If only one exists, implement `__repr__`.
4. **Why must `__eq__` and `__hash__` be consistent?**
   Equal objects must hash equally or dict/set lookups break. If you define `__eq__` and objects should stay
   hashable, define `__hash__` (or use `@dataclass(frozen=True)`).
5. **When would you use `@classmethod` over `@staticmethod`?**
   `@classmethod` receives `cls` and is the idiomatic alternative constructor (`from_statement`, `from_config`);
   `@staticmethod` is a namespaced helper with no access to instance or class state.
6. **How do generators reduce memory usage?**
   They produce values lazily, keeping only the current item (plus generator state) in memory instead of materialising
   the whole sequence. This is how you stream a file larger than RAM.
7. **What does a context manager guarantee?**
   That teardown runs even when the block raises, which is why connections, transactions, locks and temp files use
   `with`. `contextlib.contextmanager` lets you write one from a generator.
8. **What does `functools.wraps` do, and why does it matter?**
   It copies `__name__`, `__doc__`, annotations and `__wrapped__` from the wrapped function so tooling, logs and
   introspection still see the original function.
9. **Explain the GIL and its practical consequence.**
   Only one thread executes Python bytecode at a time, so threads speed up I/O-bound work but not CPU-bound work;
   for CPU work use `ProcessPoolExecutor`, C extensions that release the GIL, or a distributed engine.
10. **`is` vs `==`?**
    `is` tests identity (same object, e.g. `x is None`); `==` tests value equality via `__eq__`.
11. **Why are type hints not enforcement?**
    They are metadata for static analysis and IDEs; nothing stops you passing a `str` where `int` is annotated unless
    you add runtime validation (pydantic, `isinstance` checks, or tests).
12. **How do you make a class immutable?**
    `@dataclass(frozen=True)`, `NamedTuple`, or define `__setattr__`/properties with no setter; immutability makes
    instances safely hashable and shareable across threads and processes.

## Keep practising

* `02_python_coding_problems.ipynb` — timed coding problems with self-checking tests.
* `03_sql_interview_questions.ipynb` — SQL window functions against a SQLite sandbox.
* `04_pyspark_concepts.ipynb` — PySpark concepts, guarded so it runs without Spark installed.
* `05_interview_questions.ipynb` — a mixed question bank for mock interviews.

---

# Practice notebook: Python coding problems

The cells below are the exercises from `notebooks/02_python_coding_problems.ipynb`, merged here so the topic notebook is self-contained. They are self-checking: run them top to bottom and the assertions will confirm your answers.

# Python Coding Problems for Data Engineers

A hands-on workbook of the Python that actually gets tested in data-engineering interviews: streaming a file bigger than RAM, deduplicating at-least-once deliveries, hash joins, schema drift, sessionization, rolling metrics, and the small utility classes (LRU cache, retry with backoff) that show up in production pipelines.

It covers **14 problems** spanning all 18 classic topics: dedupe, frequency maps / group-by, multi-key sorting, top-K and top-N per group, sliding windows, two pointers and prefix sums, interval merging, binary search over partitions, generator streaming, CSV/JSON/NDJSON parsing, flattening nested JSON, in-memory hash joins, schema drift, late-arriving dedupe, sessionization, percentiles and rolling windows, LRU caches, and retry with exponential backoff.

## How to use this notebook

- **Run cells top to bottom** (`Kernel -> Restart & Run All`). Every problem is: a markdown statement, a solution cell, a test cell, then a notes cell.
- **Every solution is followed by self-checking assertions.** A cell that finishes without an exception is proof the solution is correct on the documented cases; the printed output shows the real results.
- **Standard library only.** No pandas, no PySpark, no `%pip` or `%%bash` magics, no third-party imports. Everything is deterministic and re-runnable.
- **Try before you look.** Cover the solution cell, attempt the problem, then reveal it and run the tests.
- **Notes cells** give the complexity, the edge cases, and 1-2 interview follow-ups you should be able to answer out loud.

## The problems

1. Deduplicate, count, and group without pandas
2. Multi-key sorting with key functions
3. Top-K and top-N-per-group with `heapq`
4. Rolling windows: max sum, moving average, percentile
5. Two pointers and prefix sums
6. Merge overlapping intervals
7. Binary search over sorted partitions
8. Generator-based streaming of a large file
9. Parse CSV, JSON, and NDJSON
10. Flatten nested JSON into rows
11. In-memory hash join of two datasets
12. Schema drift and late-arriving dedupe
13. Sessionize clickstream events
14. Production utilities: LRU cache and retry with backoff

The notebook closes with `## Interview questions`: 12 likely questions with concise model answers, plus runnable code that verifies several of those answers.


In [1]:
import platform
import sys

print(f"Python {sys.version.split()[0]} on {platform.system()}")
print("Standard library only: no pandas / pyspark / shell magics required.")


Python 3.13.11 on Darwin
Standard library only: no pandas / pyspark / shell magics required.


## Problem 1 - Deduplicate, count, and group without pandas

**Statement.** Implement the workhorses of in-memory aggregation with the standard library only:

1. `dedupe_preserving_order(records, key)` - keep the **first** occurrence of each key, in input order.
2. `frequency_map(items)` - the equivalent of `COUNT(*) GROUP BY item`.
3. `group_by(records, key)` - bucket rows per key, preserving first-seen group order.
4. `aggregate(rows)` - SQL-style `COUNT`, `COUNT(DISTINCT)`, `SUM`, and `AVG` per group.

**Example.**

```text
records = [{"id":"a","v":1}, {"id":"b","v":2}, {"id":"a","v":3}, {"id":"c","v":4}]
dedupe   -> [{"id":"a","v":1}, {"id":"b","v":2}, {"id":"c","v":4}]
frequency_map(["us","eu","us"]) -> {"us": 2, "eu": 1}
aggregate([("us",10.0), ("us",30.0), ("us",None)])
         -> {"us": {"rows": 3, "distinct": 2, "total": 40.0, "avg": 13.333...}}
```

**Constraints.**

- O(n) time; no pandas; do not mutate the input records.
- Preserve input order for dedupe, and first-seen order for groups.
- Decide and document null behaviour: here a null *key* is dropped, while a null *value* counts as a row but is excluded from `SUM` and `COUNT(DISTINCT)` (SQL semantics).


In [2]:
from collections import Counter, defaultdict
from collections.abc import Iterable
from typing import Any


def dedupe_preserving_order(records: Iterable[dict[str, Any]], key: str = "id") -> list[dict[str, Any]]:
    """Keep the first record seen for each value of `key`, in input order."""
    seen: set[Any] = set()
    out: list[dict[str, Any]] = []
    for record in records:
        value = record.get(key)
        if value is None or value in seen:
            continue
        seen.add(value)
        out.append(record)
    return out


def frequency_map(items: Iterable[Any]) -> dict[Any, int]:
    """COUNT(*) GROUP BY item, without pandas."""
    return dict(Counter(items))


def group_by(records: Iterable[dict[str, Any]], key: str) -> dict[Any, list[dict[str, Any]]]:
    """GROUP BY key -> list of rows, preserving first-seen group order."""
    groups: dict[Any, list[dict[str, Any]]] = defaultdict(list)
    for record in records:
        groups[record.get(key)].append(record)
    return dict(groups)


def aggregate(rows: Iterable[tuple[Any, float | None]]) -> dict[Any, dict[str, float]]:
    """COUNT / COUNT(DISTINCT) / SUM / AVG grouped by the first tuple element."""
    buckets: dict[Any, dict[str, Any]] = defaultdict(lambda: {"rows": 0, "seen": set(), "total": 0.0})
    for group, value in rows:
        bucket = buckets[group]
        bucket["rows"] += 1
        if value is not None:
            bucket["seen"].add(value)
            bucket["total"] += value
    return {
        group: {
            "rows": bucket["rows"],
            "distinct": len(bucket["seen"]),
            "total": bucket["total"],
            "avg": bucket["total"] / bucket["rows"],
        }
        for group, bucket in buckets.items()
    }


In [3]:
records = [
    {"id": "a", "v": 1},
    {"id": "b", "v": 2},
    {"id": "a", "v": 3},
    {"id": "c", "v": 4},
    {"id": "b", "v": 5},
]
print("dedupe:", dedupe_preserving_order(records))
assert dedupe_preserving_order(records) == [
    {"id": "a", "v": 1},
    {"id": "b", "v": 2},
    {"id": "c", "v": 4},
]
assert dedupe_preserving_order(records, key="v") == records  # every v is unique
assert dedupe_preserving_order([]) == []
assert dedupe_preserving_order([{"id": None}, {"id": None}, {"id": 1}]) == [{"id": 1}]

assert frequency_map(["us", "eu", "us", "us", "eu"]) == {"us": 3, "eu": 2}
assert frequency_map([]) == {}

grouped = group_by(records, key="id")
print("group order:", list(grouped))
assert list(grouped) == ["a", "b", "c"]  # first-seen order, not sorted
assert [row["v"] for row in grouped["a"]] == [1, 3]

agg = aggregate([("us", 10.0), ("eu", 5.0), ("us", 30.0), ("eu", 5.0), ("us", None)])
print("aggregate:", agg)
assert agg["us"] == {"rows": 3, "distinct": 2, "total": 40.0, "avg": 40.0 / 3}
assert agg["eu"] == {"rows": 2, "distinct": 1, "total": 10.0, "avg": 5.0}
assert aggregate([]) == {}
print("P1 checks passed")


dedupe: [{'id': 'a', 'v': 1}, {'id': 'b', 'v': 2}, {'id': 'c', 'v': 4}]
group order: ['a', 'b', 'c']
aggregate: {'us': {'rows': 3, 'distinct': 2, 'total': 40.0, 'avg': 13.333333333333334}, 'eu': {'rows': 2, 'distinct': 1, 'total': 10.0, 'avg': 5.0}}
P1 checks passed


**Complexity & edge cases.** Dedupe and counting are O(n) time; dedupe uses O(k) memory for k distinct keys, while `group_by`/`aggregate` use O(n) because they retain rows or distinct values. Edge cases: empty input returns `[]`/`{}`; unhashable keys must be normalised to tuples; `1 == True` and `1 == 1.0` collide in a Python dict, so normalise types on ingest; summing money in `float` drifts, so accumulate integer cents or `Decimal`; keep order in a list because iterating a `set` is not order-stable across runs.

**Interview follow-ups.**

1. *The data no longer fits in RAM - how does each operation change?* Sort-merge or hash-partition on disk; approximate distinct counts with HyperLogLog; a group-by becomes a shuffle in Spark.
2. *Why is `dict`/`set` lookup called O(1), and when is it not?* Average-case hashing; worst case O(n) with adversarial collisions; keys must be hashable with a stable hash.


## Problem 2 - Multi-key sorting with key functions

**Statement.** Reproduce `ORDER BY status ASC, amount DESC, id ASC`, then generalise to any number of `(field, descending)` pairs. Also handle nulls-last and the case where a key function cannot express the comparison.

**Example.**

```text
input:
  [{"status":"a","amount":3,"id":3}, {"status":"a","amount":1,"id":1},
   {"status":"b","amount":2,"id":2}, {"status":"a","amount":1,"id":4}]
output:
  [{"status":"a","amount":3,"id":3}, {"status":"a","amount":1,"id":1},
   {"status":"a","amount":1,"id":4}, {"status":"b","amount":2,"id":2}]
```

**Constraints.** Use a key function where possible; avoid `cmp_to_key` in the hot path; `None` must sort last and never raise `TypeError`; do not mutate the input list.


In [4]:
from functools import cmp_to_key
from typing import Any


def sort_orders(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """ORDER BY status ASC, amount DESC, id ASC using a single tuple key."""
    return sorted(rows, key=lambda row: (row["status"], -row["amount"], row["id"]))


def sort_by_keys_stable(rows: list[dict[str, Any]], keys: list[tuple[str, bool]]) -> list[dict[str, Any]]:
    """ORDER BY any number of (field, descending) pairs using stable passes.

    Sorting from the least-significant key to the most-significant key and
    relying on Python's stable sort matches one combined tuple key, but works
    for mixed directions and for types that cannot be negated (strings, dates).
    """
    out = list(rows)
    for field, descending in reversed(keys):
        out.sort(key=lambda row: row[field], reverse=descending)
    return out


def sort_nulls_last(rows: list[dict[str, Any]], field: str) -> list[dict[str, Any]]:
    """None sorts after every real value because the boolean flag leads the key."""
    return sorted(rows, key=lambda row: (row[field] is None, row[field]))


def sort_with_comparator(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """The escape hatch when a key function cannot express the comparison."""

    def compare(left: dict[str, Any], right: dict[str, Any]) -> int:
        if left["status"] != right["status"]:
            return -1 if left["status"] < right["status"] else 1
        if left["amount"] != right["amount"]:
            return -1 if left["amount"] > right["amount"] else 1
        return (left["id"] > right["id"]) - (left["id"] < right["id"])

    return sorted(rows, key=cmp_to_key(compare))


In [5]:
rows = [
    {"status": "a", "amount": 3, "id": 3},
    {"status": "a", "amount": 1, "id": 1},
    {"status": "b", "amount": 2, "id": 2},
    {"status": "a", "amount": 1, "id": 4},
]
expected = [
    {"status": "a", "amount": 3, "id": 3},
    {"status": "a", "amount": 1, "id": 1},
    {"status": "a", "amount": 1, "id": 4},
    {"status": "b", "amount": 2, "id": 2},
]
print("sorted:", [(r["status"], r["amount"], r["id"]) for r in sort_orders(rows)])
assert sort_orders(rows) == expected
assert sort_by_keys_stable(rows, [("status", False), ("amount", True), ("id", False)]) == expected
assert sort_with_comparator(rows) == expected

before = [dict(row) for row in rows]
sort_orders(rows)
assert rows == before, "sorted() must not mutate the input"

# Stability: equal keys keep their original relative order.
letters = [("b", 1), ("a", 1), ("b", 0)]
assert sorted(letters, key=lambda pair: pair[1]) == [("b", 0), ("b", 1), ("a", 1)]

# Nulls-last never raises, even when None sits next to a number.
assert sort_nulls_last([{"x": 2}, {"x": None}, {"x": 1}], "x") == [{"x": 1}, {"x": 2}, {"x": None}]
assert sort_nulls_last([], "x") == []
print("P2 checks passed")


sorted: [('a', 3, 3), ('a', 1, 1), ('a', 1, 4), ('b', 2, 2)]
P2 checks passed


**Complexity & edge cases.** Sorting is O(n log n) comparisons plus O(n) memory for `sorted` (`list.sort` is in place). Python's Timsort is stable, which is what makes multi-pass sorting and deterministic tie-breaking work. Edge cases: `None` inside a tuple key raises `TypeError` unless a null flag leads the tuple; negating fails for strings, dates, and custom objects, so prefer `reverse=` passes; naive and aware datetimes cannot be compared; `cmp_to_key` runs Python for every comparison and is several times slower than a key function.

**Interview follow-ups.**

1. *How would you sort 100 GB on one machine?* Sort each chunk, write it to disk, then k-way merge with `heapq.merge`; keep only one buffer per chunk in memory.
2. *Why does `sorted` need the key function to be cheap?* The key is computed once per element (decorate-sort-undecorate), but an expensive or non-deterministic key breaks that assumption; the comparator variant recomputes logic on every comparison.


## Problem 3 - Top-K and top-N-per-group with `heapq`

**Statement.** Return the K most frequent keys from a stream, and the largest N rows per group, without sorting everything. A full sort is O(n log n); a bounded heap is O(n log k) and stores only k elements.

**Example.**

```text
keys = ["a","b","a","c","a","b","d"], k = 2
top_k -> [("a", 3), ("b", 2)]

rows grouped by dept, top 2 salaries per dept, computed in one pass
eng: [("bob",150), ("ana",120)]
ops: [("ed",110),  ("di",100)]
```

**Constraints.** Return `[]` for `k <= 0`; ties must be deterministic; keep memory bounded per group; do not sort the full input.


In [6]:
import heapq
from collections import Counter, defaultdict
from collections.abc import Iterable
from typing import Any


def top_k(keys: Iterable[Any], k: int) -> list[tuple[Any, int]]:
    """K most common keys; Counter.most_common uses a heap internally."""
    return Counter(keys).most_common(k)


def top_k_bounded(keys: Iterable[Any], k: int) -> list[tuple[Any, int]]:
    """Explicit bounded min-heap: O(n log k) time, O(k) extra memory."""
    if k <= 0:
        return []
    heap: list[tuple[int, Any]] = []
    for key, count in Counter(keys).items():
        if len(heap) < k:
            heapq.heappush(heap, (count, key))
        elif count > heap[0][0]:
            heapq.heapreplace(heap, (count, key))
    return sorted(((key, count) for count, key in heap), key=lambda item: (-item[1], str(item[0])))


def top_n_per_group(
    rows: Iterable[dict[str, Any]], group: str, metric: str, n: int
) -> dict[Any, list[dict[str, Any]]]:
    """Largest n rows per group in one pass, with bounded memory per group."""
    if n <= 0:
        return {}
    heaps: dict[Any, list[tuple[float, int, dict[str, Any]]]] = defaultdict(list)
    for index, row in enumerate(rows):
        heap = heaps[row[group]]
        # The running index breaks metric ties deterministically.
        heapq.heappush(heap, (row[metric], index, row))
        if len(heap) > n:
            heapq.heappop(heap)
    return {
        group_key: [
            row for _value, _index, row in sorted(heap, key=lambda item: (-item[0], item[1]))
        ]
        for group_key, heap in heaps.items()
    }


In [7]:
keys = ["a", "b", "a", "c", "a", "b", "d"]
print("top_k:", top_k(keys, 2))
assert top_k(keys, 2) == [("a", 3), ("b", 2)]
assert top_k_bounded(keys, 2) == [("a", 3), ("b", 2)]
assert top_k(keys, 0) == []
assert top_k_bounded(keys, 0) == []
assert sorted(top_k_bounded(keys, 10)) == sorted(top_k(keys, 10))

rows = [
    {"dept": "eng", "name": "ana", "salary": 120},
    {"dept": "eng", "name": "bob", "salary": 150},
    {"dept": "eng", "name": "cy", "salary": 90},
    {"dept": "ops", "name": "di", "salary": 100},
    {"dept": "ops", "name": "ed", "salary": 110},
]
top2 = top_n_per_group(rows, "dept", "salary", 2)
print("top2 eng:", [(r["name"], r["salary"]) for r in top2["eng"]])
assert [(r["name"], r["salary"]) for r in top2["eng"]] == [("bob", 150), ("ana", 120)]
assert [(r["name"], r["salary"]) for r in top2["ops"]] == [("ed", 110), ("di", 100)]
assert top_n_per_group(rows, "dept", "salary", 0) == {}
assert top_n_per_group([], "dept", "salary", 2) == {}
print("P3 checks passed")


top_k: [('a', 3), ('b', 2)]
top2 eng: [('bob', 150), ('ana', 120)]
P3 checks passed


**Complexity & edge cases.** `heapq.nlargest(k, ...)`/`nsmallest` and `Counter.most_common(k)` are O(n log k) with O(k) memory. `top_n_per_group` is O(n log n) worst case but only O(g x n) memory for g groups. Edge cases: `k` larger than the number of distinct keys returns everything; ties need an explicit secondary key or the output is order-dependent; one heavy hitter dominates a `Counter`, so a Count-Min Sketch is the scalable answer; unhashable keys must be normalised first; `k <= 0` must not crash.

**Interview follow-ups.**

1. *When is `nlargest` worse than a full sort?* When k is close to n: O(n log k) approaches O(n log n) while adding heap overhead, so `sorted(...)[:k]` can win.
2. *How do you get the top K globally across partitions?* Each partition emits its local top K, then a single reducer merges K x partitions candidates with a bounded heap.


## Problem 4 - Rolling windows: max sum, moving average, percentile

**Statement.** Implement the windowed primitives behind dashboards, rate limiters, and SLOs:

1. `moving_average(values, window)` - trailing simple moving average, one value per full window.
2. `max_window_sum(values, window)` - largest sum of any contiguous window of exactly `window` elements.
3. `percentile(values, pct)` and `rolling_percentile(values, window, pct)` - linear-interpolation percentiles.
4. `trailing_event_count(timestamps, window)` - events in `(t - window, t]`, amortised O(1) per event.

**Example.**

```text
moving_average([1,2,3,4,5], 3)          -> [2.0, 3.0, 4.0]
max_window_sum([1,2,3,4,5], 3)          -> 12
percentile([10,12,13,15,20,100], 50)    -> 14.0
trailing_event_count([0,1,2,10,11], 3)  -> [1, 2, 3, 1, 2]
```

**Constraints.** O(1) update per element for the average and the count; document the window boundary; raise on a non-positive window; handle a window larger than the input.


In [8]:
import math
from collections import deque
from collections.abc import Sequence


def moving_average(values: Sequence[float], window: int) -> list[float]:
    """Trailing simple moving average, one output per full window."""
    if window <= 0:
        raise ValueError("window must be positive")
    if window > len(values):
        return []
    out: list[float] = []
    total = 0.0
    for index, value in enumerate(values):
        total += value
        if index >= window:
            total -= values[index - window]
        if index >= window - 1:
            out.append(total / window)
    return out


def max_window_sum(values: Sequence[float], window: int) -> float | None:
    """Largest sum of any contiguous window of exactly `window` elements."""
    if window <= 0 or window > len(values):
        return None
    total = sum(values[:window])
    best = total
    for index in range(window, len(values)):
        total += values[index] - values[index - window]
        best = max(best, total)
    return best


def percentile(values: Sequence[float], pct: float) -> float:
    """Linear-interpolation percentile (the numpy default); pct in [0, 100]."""
    if not values:
        raise ValueError("empty sample")
    if not 0 <= pct <= 100:
        raise ValueError("pct must be in [0, 100]")
    ordered = sorted(values)
    position = (len(ordered) - 1) * pct / 100
    lower = math.floor(position)
    upper = math.ceil(position)
    if lower == upper:
        return float(ordered[int(position)])
    weight = position - lower
    return ordered[lower] * (1 - weight) + ordered[upper] * weight


def rolling_percentile(values: Sequence[float], window: int, pct: float) -> list[float]:
    """Percentile of each trailing window (recomputes the sort per window)."""
    if window <= 0:
        raise ValueError("window must be positive")
    return [percentile(values[i : i + window], pct) for i in range(len(values) - window + 1)]


def trailing_event_count(timestamps: Sequence[int], window: int) -> list[int]:
    """Deque-based count of events in (t - window, t]; amortised O(1) per event."""
    buffer: deque[int] = deque()
    out: list[int] = []
    for current in timestamps:
        buffer.append(current)
        while buffer and buffer[0] <= current - window:
            buffer.popleft()
        out.append(len(buffer))
    return out


In [9]:
print("moving_average:", moving_average([1, 2, 3, 4, 5], 3))
assert moving_average([1, 2, 3, 4, 5], 3) == [2.0, 3.0, 4.0]
assert moving_average([1], 1) == [1.0]
assert moving_average([1, 2], 3) == []
try:
    moving_average([1, 2], 0)
except ValueError:
    pass
else:
    raise AssertionError("window=0 must raise")

print("max_window_sum:", max_window_sum([1, 2, 3, 4, 5], 3))
assert max_window_sum([1, 2, 3, 4, 5], 3) == 12
assert max_window_sum([-5, -1, -3], 2) == -4
assert max_window_sum([1, 2], 3) is None
assert max_window_sum([], 1) is None

print("percentile p50:", percentile([10.0, 12.0, 13.0, 15.0, 20.0, 100.0], 50))
assert percentile([1, 2, 3, 4], 50) == 2.5
assert percentile([10.0, 12.0, 13.0, 15.0, 20.0, 100.0], 50) == 14.0
assert percentile([5], 99) == 5.0
assert percentile([1.0, 2.0, 3.0], 0) == 1.0
assert percentile([1.0, 2.0, 3.0], 100) == 3.0
assert rolling_percentile([1, 2, 3, 4], 3, 100) == [3.0, 4.0]

print("trailing_event_count:", trailing_event_count([0, 1, 2, 10, 11], 3))
assert trailing_event_count([0, 1, 2, 10, 11], 3) == [1, 2, 3, 1, 2]
assert trailing_event_count([], 3) == []
print("P4 checks passed")


moving_average: [2.0, 3.0, 4.0]
max_window_sum: 12
percentile p50: 14.0
trailing_event_count: [1, 2, 3, 1, 2]
P4 checks passed


**Complexity & edge cases.** The moving average and event count are O(n) time with O(window) memory (the deque only ever holds live events); `max_window_sum` is O(n) with O(1) extra space; `rolling_percentile` is O(n x w log w) because each window is re-sorted. Edge cases: a window larger than the input yields `[]`/`None`; define whether the window is inclusive at the left edge (here it is exclusive: `t - window` is dropped); empty samples raise in `percentile`; floating-point averages can drift, so use `math.fsum` or Kahan summation for very long windows; p99 on a handful of points is just the maximum and is statistical noise.

**Interview follow-ups.**

1. *How do you compute an exact rolling median efficiently?* Keep a max-heap of the lower half and a min-heap of the upper half, rebalancing on insert/evict: O(log window) per element.
2. *Why are percentiles not additive across buckets?* The p99 of a union is not the max or average of the per-bucket p99s - you must merge the underlying sketches or raw samples.


## Problem 5 - Two pointers and prefix sums

**Statement.** Find two values in a sorted array that sum to a target (two pointers), answer range-sum queries in O(1) (prefix sums), count subarrays summing to a target (prefix sums + hash map), and find the first element at or after a target (`bisect`).

**Example.**

```text
two_sum_sorted([1,3,4,6,8,11], 10)   -> (2, 3)          # values 4 + 6
build_prefix_sums([2,4,6,8])         -> [0, 2, 6, 12, 20]
range_sum(prefix, 1, 3)              -> 10               # 4 + 6
count_subarrays_with_sum([1,1,1], 2) -> 2
first_at_or_after([1,3,3,7], 3)      -> 3
```

**Constraints.** Two pointers is O(n) time and O(1) space on already-sorted input; prefix arrays build in O(n) and answer in O(1); support negative numbers.


In [10]:
from bisect import bisect_left, bisect_right
from collections.abc import Sequence
from itertools import accumulate


def two_sum_sorted(nums: Sequence[int], target: int) -> tuple[int, int] | None:
    """Two-pointer scan on a sorted array; O(n) time, O(1) space."""
    left, right = 0, len(nums) - 1
    while left < right:
        total = nums[left] + nums[right]
        if total == target:
            return left, right
        if total < target:
            left += 1
        else:
            right -= 1
    return None


def build_prefix_sums(values: Sequence[int]) -> list[int]:
    """prefix[i] = sum(values[:i]); length n + 1 so prefix[0] == 0."""
    return [0, *accumulate(values)]


def range_sum(prefix: Sequence[int], lo: int, hi: int) -> int:
    """sum(values[lo:hi]) in O(1) once the prefix array is built."""
    if not 0 <= lo <= hi <= len(prefix) - 1:
        raise ValueError("invalid range")
    return prefix[hi] - prefix[lo]


def count_subarrays_with_sum(values: Sequence[int], target: int) -> int:
    """Count contiguous subarrays summing to target (works with negatives)."""
    seen = {0: 1}
    running = 0
    count = 0
    for value in values:
        running += value
        count += seen.get(running - target, 0)
        seen[running] = seen.get(running, 0) + 1
    return count


def first_at_or_after(sorted_values: Sequence[int], target: int) -> int | None:
    """Lower bound via bisect; None when every value is smaller than target."""
    index = bisect_left(sorted_values, target)
    return sorted_values[index] if index < len(sorted_values) else None


def last_at_or_before(sorted_values: Sequence[int], target: int) -> int | None:
    """Upper bound minus one; None when every value is larger than target."""
    index = bisect_right(sorted_values, target) - 1
    return sorted_values[index] if index >= 0 else None


In [11]:
assert two_sum_sorted([1, 3, 4, 6, 8, 11], 10) == (2, 3)
assert two_sum_sorted([1, 2, 3], 100) is None
assert two_sum_sorted([1], 1) is None
assert two_sum_sorted([-3, -1, 4, 9], 1) == (0, 2)

prefix = build_prefix_sums([2, 4, 6, 8])
print("prefix:", prefix, "range[1:3] =", range_sum(prefix, 1, 3))
assert prefix == [0, 2, 6, 12, 20]
assert range_sum(prefix, 1, 3) == 10
assert range_sum(prefix, 0, 4) == 20
assert range_sum(prefix, 2, 2) == 0
try:
    range_sum(prefix, 3, 1)
except ValueError:
    pass
else:
    raise AssertionError("lo > hi must raise")

assert count_subarrays_with_sum([1, 1, 1], 2) == 2
assert count_subarrays_with_sum([1, -1, 1, -1], 0) == 4
assert count_subarrays_with_sum([], 0) == 0

print("first_at_or_after([1,3,3,7], 3) =", first_at_or_after([1, 3, 3, 7], 3))
assert first_at_or_after([1, 3, 3, 7], 3) == 3
assert first_at_or_after([1, 3, 3, 7], 8) is None
assert first_at_or_after([], 1) is None
assert last_at_or_before([1, 3, 3, 7], 3) == 3
assert last_at_or_before([1, 3, 3, 7], 0) is None
print("P5 checks passed")


prefix: [0, 2, 6, 12, 20] range[1:3] = 10
first_at_or_after([1,3,3,7], 3) = 3
P5 checks passed


**Complexity & edge cases.** Two pointers is O(n) time / O(1) space but only valid on sorted data; a prefix array builds in O(n) and each query is O(1); `count_subarrays_with_sum` is O(n) time / O(n) memory and handles negatives (sliding windows do not). Edge cases: duplicate values are fine because any valid pair is acceptable; `lo == hi` returns 0; empty input; `bisect_left` gives the first equal element, `bisect_right - 1` gives the last; a monotonic predicate is required for binary search, so date partitions must be zero-padded ISO strings.

**Interview follow-ups.**

1. *Why does a sliding window fail with negative numbers?* Shrinking the window does not monotonically reduce the sum, so you cannot decide when to move the left edge.
2. *How do prefix sums extend to 2-D?* Build a summed-area table where `S[i][j]` is the sum of the rectangle above-left of `(i, j)`; each rectangle query is four lookups.


## Problem 6 - Merge overlapping intervals

**Statement.** Merge every overlapping or touching half-open `[start, end)` interval - how a scheduler coalesces file-availability windows and how compaction groups small files.

**Example.**

```text
merge([[1,3],[2,6],[8,10],[15,18]]) -> [[1,6],[8,10],[15,18]]
merge([[1,4],[4,5]])                -> [[1,5]]     # touching merges
merge([[1,10],[2,3],[4,5]])         -> [[1,10]]    # nested absorbed
```

**Constraints.** O(n log n) time; do not mutate the input; raise on `start > end`; keep zero-length intervals; treat touching endpoints as overlapping (half-open intervals).


In [12]:
from collections.abc import Iterable


def merge_intervals(intervals: Iterable[tuple[int, int]]) -> list[tuple[int, int]]:
    """Merge overlapping or touching half-open [start, end) intervals."""
    merged: list[tuple[int, int]] = []
    for start, end in sorted(intervals):
        if start > end:
            raise ValueError(f"invalid interval: {(start, end)}")
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    return merged


In [13]:
print(merge_intervals([(1, 3), (2, 6), (8, 10), (15, 18)]))
assert merge_intervals([(1, 3), (2, 6), (8, 10), (15, 18)]) == [(1, 6), (8, 10), (15, 18)]
assert merge_intervals([(1, 4), (4, 5)]) == [(1, 5)]
assert merge_intervals([(1, 10), (2, 3), (4, 5)]) == [(1, 10)]
assert merge_intervals([(5, 5)]) == [(5, 5)]
assert merge_intervals([]) == []

# Input order must not matter, and the input must not be mutated.
data = [(8, 10), (1, 3), (2, 6), (15, 18)]
snapshot = list(data)
assert merge_intervals(data) == [(1, 6), (8, 10), (15, 18)]
assert data == snapshot

try:
    merge_intervals([(3, 1)])
except ValueError:
    pass
else:
    raise AssertionError("start > end must raise")
print("P6 checks passed")


[(1, 6), (8, 10), (15, 18)]
P6 checks passed


**Complexity & edge cases.** O(n log n) for the sort plus O(n) for the sweep and the output; `sorted` copies, so the input is untouched. Edge cases: empty input returns `[]`; zero-length intervals survive; closed intervals need `start < merged[-1][1]` so touching endpoints stay separate; nested intervals are absorbed by `max`; huge coordinates favour an interval list over a bitmap; streaming intervals that arrive unordered require a buffer and can only be partially merged.

**Interview follow-ups.**

1. *How would you find the peak number of concurrent intervals?* Sweep the endpoints as `(+1, start)` and `(-1, end)` events, sort, and track a running maximum - O(n log n), or O(n) counting sort on small integer ranges.
2. *How does this relate to small-file compaction?* Group adjacent files whose size range overlaps or is below a threshold, then rewrite each group as one file - the same merge-with-tolerance idea.


## Problem 7 - Binary search over sorted partitions

**Statement.** A table is stored as sorted ISO-date partitions. (a) Find the partition that can contain a key, or the first partition at or after a watermark. (b) Generalise the partitioning idea: find the k-th smallest element of two sorted arrays by binary-searching the split point, which is the "median of two sorted arrays" interview classic.

**Example.**

```text
partitions = ["2026-01-01","2026-02-01","2026-03-01","2026-04-01"]
partition_for_key("2026-03-15")   -> "2026-03-01"
first_at_or_after("2026-03-15")   -> "2026-04-01"

kth_smallest_two_sorted([1,3,8], [2,4,5,6,7], k=4) -> 4
```

**Constraints.** O(log n) per lookup; O(log(min(n, m))) for the two-array selection; handle empty inputs and out-of-range `k`; the sorted arrays must be ascending (raise if not).


In [14]:
from bisect import bisect_left, bisect_right
from collections.abc import Sequence


def partition_for_key(partitions: Sequence[str], key: str) -> str | None:
    """Last partition whose start is <= key (the file that can contain key)."""
    index = bisect_right(partitions, key) - 1
    return partitions[index] if index >= 0 else None


def first_at_or_after(partitions: Sequence[str], watermark: str) -> str | None:
    """First partition at or after a watermark (where incremental processing resumes)."""
    index = bisect_left(partitions, watermark)
    return partitions[index] if index < len(partitions) else None


def kth_smallest_two_sorted(a: Sequence[int], b: Sequence[int], k: int) -> int:
    """k-th smallest (1-indexed) of two sorted arrays via partition binary search."""
    if not 1 <= k <= len(a) + len(b):
        raise ValueError("k out of range")
    if any(a[i] > a[i + 1] for i in range(len(a) - 1)) or any(b[i] > b[i + 1] for i in range(len(b) - 1)):
        raise ValueError("inputs must be sorted")
    if len(a) > len(b):
        a, b = b, a  # binary-search the shorter array
    n, m = len(a), len(b)
    low, high = max(0, k - m), min(k, n)
    while low <= high:
        i = (low + high) // 2          # take i elements from a
        j = k - i                      # ... and j elements from b
        a_left = a[i - 1] if i > 0 else float("-inf")
        a_right = a[i] if i < n else float("inf")
        b_left = b[j - 1] if j > 0 else float("-inf")
        b_right = b[j] if j < m else float("inf")
        if a_left <= b_right and b_left <= a_right:
            return max(a_left, b_left)
        if a_left > b_right:
            high = i - 1
        else:
            low = i + 1
    raise RuntimeError("unreachable: inputs were not sorted")


def median_two_sorted(a: Sequence[int], b: Sequence[int]) -> float:
    """Median of two sorted arrays built on the k-th selection."""
    total = len(a) + len(b)
    if total == 0:
        raise ValueError("both arrays are empty")
    if total % 2 == 1:
        return float(kth_smallest_two_sorted(a, b, total // 2 + 1))
    lower = kth_smallest_two_sorted(a, b, total // 2)
    upper = kth_smallest_two_sorted(a, b, total // 2 + 1)
    return (lower + upper) / 2


In [15]:
partitions = ["2026-01-01", "2026-02-01", "2026-03-01", "2026-04-01"]
print("partition_for_key:", partition_for_key(partitions, "2026-03-15"))
assert partition_for_key(partitions, "2026-03-15") == "2026-03-01"
assert partition_for_key(partitions, "2026-01-01") == "2026-01-01"
assert partition_for_key(partitions, "2025-12-31") is None
assert partition_for_key([], "2026-01-01") is None
assert first_at_or_after(partitions, "2026-03-15") == "2026-04-01"
assert first_at_or_after(partitions, "2026-03-01") == "2026-03-01"
assert first_at_or_after(partitions, "2027-01-01") is None

a, b = [1, 3, 8], [2, 4, 5, 6, 7]
merged = sorted(a + b)
print("merged:", merged)
for k in range(1, len(merged) + 1):
    assert kth_smallest_two_sorted(a, b, k) == merged[k - 1]
assert kth_smallest_two_sorted([], [1, 2, 3], 2) == 2
assert kth_smallest_two_sorted([1, 2], [], 1) == 1
assert median_two_sorted([1, 3], [2]) == 2.0
assert median_two_sorted([1, 2], [3, 4]) == 2.5

for bad_call in (
    lambda: kth_smallest_two_sorted([1], [2], 3),
    lambda: kth_smallest_two_sorted([2, 1], [3], 2),
    lambda: median_two_sorted([], []),
):
    try:
        bad_call()
    except ValueError:
        pass
    else:
        raise AssertionError("invalid input must raise ValueError")
print("P7 checks passed")


partition_for_key: 2026-03-01
merged: [1, 2, 3, 4, 5, 6, 7, 8]
P7 checks passed


**Complexity & edge cases.** `bisect` gives O(log n) lookups in C; the two-array selection is O(log(min(n, m))) time and O(1) space. Edge cases: empty partition lists; duplicate partition starts (`bisect_left` -> first, `bisect_right` -> last); lexicographic order only equals chronological order for zero-padded ISO dates; `k` out of range; one empty array; even total length needs the average of two order statistics; unsorted input must be rejected, because the partition invariant silently breaks.

**Interview follow-ups.**

1. *Why binary-search the shorter array?* It bounds the search range at min(n, m) and keeps the partition indices valid, giving the tighter O(log(min(n,m))) bound.
2. *How do you binary-search data in a data lake?* Store per-file min/max statistics (Parquet footers, Iceberg/Delta manifests) and binary-search the sorted manifest instead of scanning every file.


## Problem 8 - Generator-based streaming of a large file

**Statement.** Process a file far larger than RAM on one machine. Read it lazily, transform in bounded batches, and never hold the whole file. Generators keep memory flat.

**Example.**

```text
file   : events.csv.gz with lines "user_id,amount" (millions of rows)
target : total amount per user, O(distinct users) memory
search : first N matching lines, short-circuiting the read
```

**Constraints.** O(1) memory for the reader, independent of file size; handle plain and gzip files; skip blank lines; dead-letter malformed rows instead of aborting; no third-party libraries.


In [16]:
import gzip
from collections import defaultdict
from collections.abc import Iterator
from pathlib import Path

try:  # Python 3.12+
    from itertools import batched
except ImportError:  # Python 3.11 and older, including some Colab runtimes
    from itertools import islice

    def batched(iterable, n):
        "Batch data into tuples of length n; the last batch may be shorter."
        it = iter(iterable)
        while chunk := tuple(islice(it, n)):
            yield chunk


def read_lines(path: Path) -> Iterator[str]:
    """Lazily yield stripped lines, transparently handling .gz files."""
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as handle:
        for line in handle:
            yield line.rstrip("\r\n")


def totals_by_user(path: Path) -> dict[str, float]:
    """One streaming pass: O(batch + distinct users) memory, not O(file)."""
    totals: dict[str, float] = defaultdict(float)
    for batch in batched(read_lines(path), 1_000):
        for line in batch:
            if not line:
                continue
            user_id, _, amount = line.partition(",")
            try:
                value = float(amount)
            except ValueError:
                continue  # dead-letter malformed rows instead of aborting
            # Add only after the cast succeeds: `totals[user_id] += ...` would
            # create a 0.0 entry via defaultdict before the cast raises.
            totals[user_id] += value
    return dict(totals)


def first_n_matching(path: Path, needle: str, n: int) -> list[str]:
    """Short-circuiting search over a huge file: stop as soon as n matches exist."""
    matches: list[str] = []
    for line in read_lines(path):
        if needle in line:
            matches.append(line)
            if len(matches) == n:
                break
    return matches


In [17]:
import gzip
import itertools
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)
    raw = tmp_path / "events.csv"
    raw.write_text(
        "u1,10\n\n" + "u2,5\n" + "u1,2.5\n" + "bad,not-a-number\n" + "u3,7",
        encoding="utf-8",
    )
    print("totals:", totals_by_user(raw))
    assert totals_by_user(raw) == {"u1": 12.5, "u2": 5.0, "u3": 7.0}
    assert first_n_matching(raw, "u1", 1) == ["u1,10"]
    assert first_n_matching(raw, "missing", 5) == []

    gz = tmp_path / "events.csv.gz"
    with gzip.open(gz, "wt", encoding="utf-8") as handle:
        handle.write("a,1\nb,2\n")
    assert totals_by_user(gz) == {"a": 1.0, "b": 2.0}


def naturals():
    value = 0
    while True:
        yield value
        value += 1


# Generators are lazy: an infinite source is fine as long as you stop early.
assert list(itertools.islice(naturals(), 3)) == [0, 1, 2]
gen = naturals()
assert next(gen) == 0 and next(gen) == 1  # state advances only on demand


def explode():
    yield 1
    raise AssertionError("generator body must not run eagerly")


probe = explode()
assert next(probe) == 1  # nothing raises until the generator is advanced
probe.close()
print("P8 checks passed")


totals: {'u1': 12.5, 'u2': 5.0, 'u3': 7.0}
P8 checks passed


**Complexity & edge cases.** O(n) time, O(batch size + retained state) memory; the reader holds one line at a time. Edge cases: a final line without a newline; CRLF endings; malformed records should be dead-lettered with a reason rather than killing the job; sniff gzip magic bytes instead of trusting the suffix in production; parallel workers need byte ranges that resync to the next newline; closing a generator early still runs the `with` block via `GeneratorExit`.

**Interview follow-ups.**

1. *How do you parallelise a single huge file?* Give each worker a byte range, seek, discard the partial first line, and stop at the last newline; gzip blocks can be decompressed in parallel with `zlib`-level offsets.
2. *Generator vs `itertools.batched` list chunks - what changes?* Batching gives amortised per-item overhead and lets you hand whole chunks to a vectorised or bulk-insert path, at the cost of holding one batch in memory.


## Problem 9 - Parse CSV, JSON, and NDJSON

**Statement.** Read the common landing formats without pandas: header CSV (with quoted fields and a BOM), a JSON array, NDJSON (one object per line), and a memory-flat streaming JSON array via `JSONDecoder.raw_decode`.

**Example.**

```text
orders.csv   : order_id,amount,note  /  1,10.5,"a, b"
orders.json  : [{"order_id": 1, "amount": 10.5}]
orders.ndjson: {"order_id": 1}\n{"order_id": 2}
```

**Constraints.** O(1) memory for CSV/NDJSON; report bad NDJSON with a line number; reject a non-array JSON document in `parse_json_array`; strip a UTF-8 BOM; no pandas.


In [18]:
import csv
import io
import json
from collections.abc import Iterator
from typing import Any


def parse_csv(text: str) -> list[dict[str, str]]:
    """Header-based CSV; csv handles quoted fields that contain commas/newlines."""
    return list(csv.DictReader(io.StringIO(text.lstrip("\ufeff"))))


def parse_json_array(text: str) -> list[dict[str, Any]]:
    """A JSON array must be materialised unless you stream it with raw_decode."""
    payload = json.loads(text)
    if not isinstance(payload, list):
        raise ValueError("expected a JSON array")
    return payload


def parse_ndjson(text: str) -> list[dict[str, Any]]:
    """One JSON object per line; blank lines skipped, bad lines reported with a line number."""
    records: list[dict[str, Any]] = []
    for number, line in enumerate(text.splitlines(), start=1):
        if not line.strip():
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as exc:
            raise ValueError(f"bad NDJSON at line {number}") from exc
    return records


def iter_json_array(text: str) -> Iterator[Any]:
    """Stream a JSON array value-by-value with raw_decode, so it need not fit in memory.

    Real code feeds this from a chunked file reader instead of a single string;
    the point is that `raw_decode` parses one value and tells you where it stopped.
    """
    decoder = json.JSONDecoder()
    position = 0
    length = len(text)
    while position < length:
        while position < length and text[position] in " \t\r\n,[]":
            position += 1
        if position >= length:
            break
        value, position = decoder.raw_decode(text, position)
        yield value


In [19]:
csv_text = 'order_id,amount,note\n1,10.5,"a, b"\n2,3.0,\n'
print("csv:", parse_csv(csv_text))
assert parse_csv(csv_text) == [
    {"order_id": "1", "amount": "10.5", "note": "a, b"},
    {"order_id": "2", "amount": "3.0", "note": ""},
]
assert parse_csv("\ufeffid,v\n1,2\n") == [{"id": "1", "v": "2"}]  # BOM stripped

json_text = '[{"order_id": 1, "amount": 10.5}, {"order_id": 2, "amount": 3.0}]'
assert parse_json_array(json_text) == [
    {"order_id": 1, "amount": 10.5},
    {"order_id": 2, "amount": 3.0},
]
try:
    parse_json_array('{"order_id": 1}')
except ValueError:
    pass
else:
    raise AssertionError("non-array JSON must raise")

ndjson_text = '{"order_id": 1}\n\n{"order_id": 2}\n'
assert parse_ndjson(ndjson_text) == [{"order_id": 1}, {"order_id": 2}]
assert parse_ndjson("") == []
try:
    parse_ndjson('{"ok": 1}\nnot json\n')
except ValueError as exc:
    assert "line 2" in str(exc)
else:
    raise AssertionError("bad NDJSON must raise")

streamed = list(iter_json_array('  [ {"a": 1}, {"b": [1, 2]}, 3 ] '))
print("streamed JSON:", streamed)
assert streamed == [{"a": 1}, {"b": [1, 2]}, 3]
assert list(iter_json_array("[]")) == []
print("P9 checks passed")


csv: [{'order_id': '1', 'amount': '10.5', 'note': 'a, b'}, {'order_id': '2', 'amount': '3.0', 'note': ''}]
streamed JSON: [{'a': 1}, {'b': [1, 2]}, 3]
P9 checks passed


**Complexity & edge cases.** CSV and NDJSON are O(n) time / O(1) memory; a JSON array is O(n) memory because the document must exist. `raw_decode` makes the array streamable but requires you to skip separators yourself. Edge cases: a UTF-8 BOM (`utf-8-sig`); embedded newlines inside quoted CSV fields; very long NDJSON lines; empty strings that break `float`; a JSON file that is an object rather than an array; deeply nested JSON needs `ijson` for true streaming.

**Interview follow-ups.**

1. *Why is NDJSON the preferred landing format for streams?* Each line is an independent, appendable, partially readable document, so a crash mid-file loses at most one line and readers can parallelise by line ranges.
2. *How do you make parsing robust to schema drift?* Parse into `dict`, coerce types with an explicit per-column cast function, and route cast failures to a dead-letter path with the raw line and error.


## Problem 10 - Flatten nested JSON into rows

**Statement.** Turn nested documents into tabular rows: (a) `flatten_record` collapses nested objects into dotted paths and indexes arrays; (b) `explode_rows` emits one output row per array element with the parent fields repeated - the in-memory equivalent of `UNNEST` / `LATERAL VIEW explode`.

**Example.**

```text
input : {"order_id":1,"customer":{"country":"us"},
         "items":[{"sku":"a","qty":2},{"sku":"b","qty":1}]}
flat  : {"order_id":1,"customer.country":"us",
         "items[0].sku":"a","items[0].qty":2,"items[1].sku":"b","items[1].qty":1}
rows  : [{"order_id":1,"customer":{...},"sku":"a","qty":2},
         {"order_id":1,"customer":{...},"sku":"b","qty":1}]
```

**Constraints.** Preserve key order; do not mutate the input; an empty array emits one parent row (left-join semantics); a missing array field behaves like an empty array.


In [20]:
from collections.abc import Iterable
from typing import Any


def flatten_record(record: dict[str, Any], prefix: str = "", sep: str = ".") -> dict[str, Any]:
    """Collapse nested objects into dotted paths; arrays are indexed."""
    out: dict[str, Any] = {}
    for key, value in record.items():
        path = f"{prefix}{sep}{key}" if prefix else key
        if isinstance(value, dict):
            out.update(flatten_record(value, path, sep))
        elif isinstance(value, list):
            for index, element in enumerate(value):
                if isinstance(element, dict):
                    out.update(flatten_record(element, f"{path}[{index}]", sep))
                else:
                    out[f"{path}[{index}]"] = element
        else:
            out[path] = value
    return out


def explode_rows(records: Iterable[dict[str, Any]], array_field: str) -> list[dict[str, Any]]:
    """One output row per array element, repeating the parent fields (UNNEST)."""
    rows: list[dict[str, Any]] = []
    for record in records:
        values = record.get(array_field) or []
        base = {key: value for key, value in record.items() if key != array_field}
        if not values:
            rows.append(dict(base))  # left-join semantics: keep one row
            continue
        for value in values:
            row = dict(base)
            if isinstance(value, dict):
                row.update(value)
            else:
                row[array_field] = value
            rows.append(row)
    return rows


In [21]:
order = {
    "order_id": 1,
    "customer": {"country": "us", "city": "austin"},
    "items": [{"sku": "a", "qty": 2}, {"sku": "b", "qty": 1}],
    "tags": ["gift", "rush"],
}
flat = flatten_record(order)
print("flat:", flat)
assert flat == {
    "order_id": 1,
    "customer.country": "us",
    "customer.city": "austin",
    "items[0].sku": "a",
    "items[0].qty": 2,
    "items[1].sku": "b",
    "items[1].qty": 1,
    "tags[0]": "gift",
    "tags[1]": "rush",
}
assert flatten_record({}) == {}
assert flatten_record({"a": {"b": {"c": 1}}}) == {"a.b.c": 1}
assert flatten_record({"a": 1}, sep="/") == {"a": 1}

rows = explode_rows([order], "items")
print("exploded rows:", len(rows))
assert rows == [
    {
        "order_id": 1,
        "customer": {"country": "us", "city": "austin"},
        "tags": ["gift", "rush"],
        "sku": "a",
        "qty": 2,
    },
    {
        "order_id": 1,
        "customer": {"country": "us", "city": "austin"},
        "tags": ["gift", "rush"],
        "sku": "b",
        "qty": 1,
    },
]
assert explode_rows([{"id": 1, "items": []}], "items") == [{"id": 1}]
assert explode_rows([{"id": 1}], "items") == [{"id": 1}]
assert explode_rows([{"id": 1, "items": ["x", "y"]}], "items") == [
    {"id": 1, "items": "x"},
    {"id": 1, "items": "y"},
]
assert explode_rows([], "items") == []
assert order["items"] == [{"sku": "a", "qty": 2}, {"sku": "b", "qty": 1}]  # not mutated
print("P10 checks passed")


flat: {'order_id': 1, 'customer.country': 'us', 'customer.city': 'austin', 'items[0].sku': 'a', 'items[0].qty': 2, 'items[1].sku': 'b', 'items[1].qty': 1, 'tags[0]': 'gift', 'tags[1]': 'rush'}
exploded rows: 2
P10 checks passed


**Complexity & edge cases.** Both functions are O(total nodes) time; `flatten_record` uses O(depth) recursion and `explode_rows` materialises one row per array element. Edge cases: a missing parent key; an empty array should still emit one parent row to match a left join; arrays of arrays and null array elements; a literal key containing the separator collides with a nested path, so production code should escape it; deeply nested documents can exceed the recursion limit, so an explicit stack is safer.

**Interview follow-ups.**

1. *Flatten or explode - how do you choose?* Flatten when you need a wide, one-row-per-document table; explode when downstream aggregation needs one row per child event, accepting the row multiplication.
2. *How does row multiplication affect downstream counts?* A naive `COUNT(*)` after exploding counts child records, not orders - count distinct parent ids or pre-aggregate before the explode.


## Problem 11 - In-memory hash join of two datasets

**Statement.** Join a large fact stream to a small dimension without a database. Build a hash map on the smaller side and stream the larger side through it - the same shape as a broadcast join. Support inner and left joins, duplicate build keys, and SQL null semantics.

**Example.**

```text
facts      : [("c1","o1"), ("c2","o2"), ("c9","o3")]
dimensions : [("c1","Alice"), ("c2","Bob")]
inner      : [("c1","o1","Alice"), ("c2","o2","Bob")]
left       : [..., ("c9","o3",None)]
```

**Constraints.** O(n + m) time, O(m) memory for the build side; always build on the smaller side; duplicate build keys multiply rows; NULL keys never match; reject unknown join types.


In [22]:
from collections import defaultdict
from collections.abc import Iterable, Iterator
from typing import Any, TypeVar

K = TypeVar("K")
V = TypeVar("V")


def hash_join(
    facts: Iterable[tuple[K, V]],
    dimensions: Iterable[tuple[K, Any]],
    how: str = "inner",
    null: Any = None,
) -> list[tuple[K, V, Any]]:
    """Broadcast-style join: build a hash index on the right (small) side."""
    if how not in {"inner", "left"}:
        raise ValueError("how must be 'inner' or 'left'")
    index: dict[K, list[Any]] = defaultdict(list)
    for key, value in dimensions:
        if key is not None:  # SQL semantics: NULL keys never match
            index[key].append(value)
    out: list[tuple[K, V, Any]] = []
    for key, fact_value in facts:
        matches = index.get(key) if key is not None else None
        if matches:
            for dim_value in matches:
                out.append((key, fact_value, dim_value))
        elif how == "left":
            out.append((key, fact_value, null))
    return out


def hash_join_streaming(
    facts: Iterable[tuple[K, V]],
    dimensions: Iterable[tuple[K, Any]],
    how: str = "inner",
) -> Iterator[tuple[K, V, Any]]:
    """Same join as a generator, so the fact side is never materialised."""
    index: dict[K, list[Any]] = defaultdict(list)
    for key, value in dimensions:
        if key is not None:
            index[key].append(value)
    for key, fact_value in facts:
        matches = index.get(key) if key is not None else None
        if matches:
            for dim_value in matches:
                yield key, fact_value, dim_value
        elif how == "left":
            yield key, fact_value, None


In [23]:
facts = [("c1", "o1"), ("c2", "o2"), ("c9", "o3")]
dims = [("c1", "Alice"), ("c2", "Bob")]
print("inner:", hash_join(facts, dims))
assert hash_join(facts, dims) == [("c1", "o1", "Alice"), ("c2", "o2", "Bob")]
assert hash_join(facts, dims, how="left") == [
    ("c1", "o1", "Alice"),
    ("c2", "o2", "Bob"),
    ("c9", "o3", None),
]

# Duplicate build-side keys multiply rows (SQL join semantics).
assert hash_join([("c1", "o1")], [("c1", "A"), ("c1", "B")]) == [
    ("c1", "o1", "A"),
    ("c1", "o1", "B"),
]

# NULL keys never match, but a left join still emits the fact row.
assert hash_join([(None, "o1")], [(None, "A")]) == []
assert hash_join([(None, "o1")], [(None, "A")], how="left") == [(None, "o1", None)]

assert list(hash_join_streaming(facts, dims)) == [("c1", "o1", "Alice"), ("c2", "o2", "Bob")]
assert hash_join([], dims) == []
assert hash_join(facts, []) == []
try:
    hash_join([], [], how="outer")
except ValueError:
    pass
else:
    raise AssertionError("unsupported join type must raise")
print("P11 checks passed")


inner: [('c1', 'o1', 'Alice'), ('c2', 'o2', 'Bob')]
P11 checks passed


**Complexity & edge cases.** O(n + m) time and O(m) memory for the m-row build side; building on the large side silently materialises the fact table. Duplicate build keys multiply rows, which is correct SQL semantics but can explode output. Edge cases: NULL keys never match in SQL (unlike `==` in Python); `1` vs `"1"` key types silently drop rows, so normalise on ingest; a single skewed key defeats the in-memory approach and needs a sort-merge or distributed shuffle.

**Interview follow-ups.**

1. *Why does Spark broadcast the small side?* It ships one copy of the small table to every executor, turning a shuffle join into a map-side join with no network shuffle of the fact data.
2. *When is a sort-merge join better than a hash join?* When neither side fits in memory, when both inputs are already sorted, or when the data is skewed and a range partition balances better than a hash.


## Problem 12 - Schema drift and late-arriving dedupe

**Statement.** Two batch-ingestion safety checks. (a) Infer the schema per batch and diff it against a baseline, classifying columns as added, removed, or retyped. (b) Deduplicate replays and out-of-order backfills by keeping the record with the greatest `(updated_at, tiebreak)` per business key.

**Example.**

```text
baseline: {"id":"int", "amount":"float"}
current : {"id":"int", "amount":"str", "coupon":"str"}
drift   : {"added":["coupon"], "removed":[], "retyped":["amount"]}

input : [("u1",t1), ("u2",t1), ("u1",t3), ("u1",t2)]
output: u1 -> the t3 record, u2 -> the t1 record
```

**Constraints.** O(rows x columns) to infer, O(columns) to diff; O(n) time / O(k) memory for dedupe where k is the distinct key count; the tiebreak makes replays deterministic; do not mutate inputs.


In [24]:
from collections.abc import Iterable, Mapping
from typing import Any


def infer_schema(rows: Iterable[Mapping[str, Any]]) -> dict[str, set[str]]:
    """Column -> set of observed Python type names (None contributes nothing)."""
    schema: dict[str, set[str]] = {}
    for row in rows:
        for column, value in row.items():
            if value is not None:
                schema.setdefault(column, set()).add(type(value).__name__)
    return schema


def describe_type(types: set[str]) -> str:
    """Collapse a set of observed types into one readable label."""
    if not types:
        return "null"
    if len(types) == 1:
        return next(iter(types))
    return "mixed(" + "|".join(sorted(types)) + ")"


def detect_schema_drift(
    baseline: Mapping[str, set[str]], current: Mapping[str, set[str]]
) -> dict[str, Any]:
    """Classify every column as added, removed, or retyped."""
    base, now = set(baseline), set(current)
    return {
        "added": sorted(now - base),
        "removed": sorted(base - now),
        "retyped": sorted(
            column
            for column in base & now
            if describe_type(baseline[column]) != describe_type(current[column])
        ),
    }


def dedupe_latest(
    records: Iterable[dict[str, Any]],
    key: str = "id",
    ts: str = "updated_at",
    tiebreak: str = "offset",
) -> list[dict[str, Any]]:
    """Keep the record with the greatest (timestamp, tiebreak) per key."""
    winners: dict[Any, dict[str, Any]] = {}
    for record in records:
        current = winners.get(record[key])
        rank = (record[ts], record.get(tiebreak, 0))
        if current is None or rank > (current[ts], current.get(tiebreak, 0)):
            winners[record[key]] = record
    return list(winners.values())


In [25]:
baseline = [{"id": 1, "amount": 10.0, "note": "x"}, {"id": 2, "amount": 3.0, "note": None}]
current = [{"id": 3, "amount": "10", "note": "y", "coupon": "SAVE"}]
drift = detect_schema_drift(infer_schema(baseline), infer_schema(current))
print("drift:", drift)
assert drift == {"added": ["coupon"], "removed": [], "retyped": ["amount"]}
assert describe_type(set()) == "null"
assert describe_type({"int"}) == "int"
assert describe_type({"float", "int"}) == "mixed(float|int)"
assert detect_schema_drift({}, {}) == {"added": [], "removed": [], "retyped": []}
assert detect_schema_drift({"a": {"int"}}, {"a": {"int"}, "b": {"str"}})["added"] == ["b"]

records = [
    {"id": "u1", "updated_at": 1, "offset": 1, "v": "old"},
    {"id": "u2", "updated_at": 1, "offset": 1, "v": "only"},
    {"id": "u1", "updated_at": 3, "offset": 9, "v": "new"},
    {"id": "u1", "updated_at": 3, "offset": 4, "v": "older-tie"},
    {"id": "u1", "updated_at": 2, "offset": 50, "v": "stale"},
]
result = dedupe_latest(records)
by_id = {row["id"]: row for row in result}
print("winners:", {k: v["v"] for k, v in by_id.items()})
assert by_id["u1"]["v"] == "new"      # greatest timestamp wins
assert by_id["u2"]["v"] == "only"     # single record retained
assert len(result) == 2
assert dedupe_latest([]) == []
assert records[0]["v"] == "old"       # input not mutated
print("P12 checks passed")


drift: {'added': ['coupon'], 'removed': [], 'retyped': ['amount']}
winners: {'u1': 'new', 'u2': 'only'}
P12 checks passed


**Complexity & edge cases.** Inference is O(rows x columns); the diff is O(columns), which is why diffing declared Parquet/Avro metadata beats sampling data. Dedupe is O(n) time / O(k) memory; a warehouse `MERGE` or sort-merge spills instead. Edge cases: all-null columns have no type; int-to-float widening is usually benign while string-to-number is breaking; column-order changes are invisible to a set diff but break positional CSV readers; equal timestamps need a version or Kafka-offset tiebreak; naive and aware datetimes cannot be compared, so normalise to UTC; tombstone/delete rows must win and be applied by the writer.

**Interview follow-ups.**

1. *Where should drift detection run - in the producer or the consumer?* Both: producers validate against a contract, consumers compare the landed batch against the catalog because producers lie or get bypassed.
2. *Why is the tiebreak important for idempotency?* Without it, two replays of the same timestamp can pick different winners, so the sink is non-deterministic and reruns do not converge.


## Problem 13 - Sessionize clickstream events

**Statement.** Group ordered user events into sessions, starting a new session when the idle gap exceeds a threshold - the exercise behind every product-analytics model. Handle out-of-order input and per-user interleaving.

**Example.**

```text
events: u1 at [0, 300, 2400, 2500], u2 at [0], gap = 1800 seconds
output: [u1 0-300 (2 events), u1 2400-2500 (2 events), u2 0-0 (1 event)]
```

**Constraints.** Sort by `(user, timestamp)` first; O(n log n) time, O(sessions) output; a gap of exactly the threshold stays in the session; reject a negative gap; a single event forms a one-event session.


In [26]:
from collections.abc import Iterable
from typing import Any


def sessionize(
    events: Iterable[dict[str, Any]],
    gap_seconds: int,
    user_key: str = "user",
    ts_key: str = "ts",
    id_key: str = "event_id",
) -> list[dict[str, Any]]:
    """Group events into sessions; a new session starts when the idle gap exceeds
    `gap_seconds`. The boundary is inclusive: a gap of exactly `gap_seconds`
    stays in the same session.
    """
    if gap_seconds < 0:
        raise ValueError("gap_seconds must be non-negative")
    ordered = sorted(events, key=lambda event: (event[user_key], event[ts_key], event[id_key]))
    sessions: list[dict[str, Any]] = []
    for event in ordered:
        user, when = event[user_key], event[ts_key]
        if (
            sessions
            and sessions[-1]["user"] == user
            and when - sessions[-1]["end"] <= gap_seconds
        ):
            session = sessions[-1]
            session["end"] = when
            session["count"] += 1
            session["duration"] = session["end"] - session["start"]
            session["event_ids"].append(event[id_key])
        else:
            sessions.append(
                {
                    "user": user,
                    "start": when,
                    "end": when,
                    "count": 1,
                    "duration": 0,
                    "event_ids": [event[id_key]],
                }
            )
    return sessions


In [27]:
events = [
    {"user": "u1", "ts": 0, "event_id": "e1"},
    {"user": "u1", "ts": 300, "event_id": "e2"},
    {"user": "u1", "ts": 2400, "event_id": "e3"},  # 2100s gap -> new session
    {"user": "u1", "ts": 2500, "event_id": "e4"},
    {"user": "u2", "ts": 0, "event_id": "e5"},
]
sessions = sessionize(events, gap_seconds=1800)
print("sessions:", [(s["user"], s["start"], s["end"], s["count"]) for s in sessions])
assert [(s["user"], s["start"], s["end"], s["count"]) for s in sessions] == [
    ("u1", 0, 300, 2),
    ("u1", 2400, 2500, 2),
    ("u2", 0, 0, 1),
]
assert sessions[0]["event_ids"] == ["e1", "e2"]
assert sessions[0]["duration"] == 300
assert sessions[2]["duration"] == 0

# Exactly the gap stays in the same session (inclusive boundary).
same = sessionize(
    [{"user": "u", "ts": 0, "event_id": "a"}, {"user": "u", "ts": 1800, "event_id": "b"}], 1800
)
assert len(same) == 1
assert same[0]["count"] == 2

# Out-of-order input is sorted before sessionizing, so the result is identical.
shuffled = [events[3], events[0], events[1], events[2], events[4]]
assert sessionize(shuffled, 1800) == sessions
assert sessionize([], 1800) == []

try:
    sessionize([], -1)
except ValueError:
    pass
else:
    raise AssertionError("negative gap must raise")
print("P13 checks passed")


sessions: [('u1', 0, 300, 2), ('u1', 2400, 2500, 2), ('u2', 0, 0, 1)]
P13 checks passed


**Complexity & edge cases.** O(n log n) if the input must be sorted, O(n) if the source already delivers ordered partitions per user; memory is O(sessions) for the output (or O(1) state per active user when streaming). Edge cases: out-of-order events need a lateness buffer before emitting; one event forms a one-event session; define `<=` vs `<` at the boundary (here exactly the gap is inactive); users with no events are absent, so downstream joins must be left joins; clock skew can make a user's timestamps non-monotonic.

**Interview follow-ups.**

1. *How would you sessionize an unbounded stream?* Keep per-user last-event state in a keyed store (Redis/Flink state) and emit a session when a timer for `last_event + gap` fires, handling late events with allowed lateness.
2. *How does this map to a Spark window function?* `LAG(ts)` over `PARTITION BY user ORDER BY ts`, then mark a new session when `ts - lag > gap`, and finally `SUM(new_session)` as a session id.


## Problem 14 - Production utilities: LRU cache and retry with backoff

**Statement.** Two small classes you will be asked to write (or reason about) in a data-platform interview: a fixed-capacity **LRU cache** with O(1) `get`/`put`, and a **retry** helper with capped exponential backoff plus optional jitter.

**Example.**

```text
capacity 2: put("a",1) put("b",2) get("a") -> 1; put("c",3) evicts "b"
backoff  : attempts=4, base=0.1, factor=2 -> sleeps [0.1, 0.2, 0.4]
```

**Constraints.** O(1) amortised `get`/`put`; validate capacity; injectable `sleep`/`rand` so the retry logic is testable and deterministic; cap both the delay and the attempt count; re-raise the final error.


In [28]:
import random
import time
from collections import OrderedDict
from collections.abc import Callable, Hashable
from typing import Generic, TypeVar

K = TypeVar("K", bound=Hashable)
V = TypeVar("V")


class LRUCache(Generic[K, V]):
    """O(1) get/put: an OrderedDict is a hash map plus a doubly linked list."""

    def __init__(self, capacity: int) -> None:
        if capacity <= 0:
            raise ValueError("capacity must be positive")
        self.capacity = capacity
        self._data: OrderedDict[K, V] = OrderedDict()
        self.hits = 0
        self.misses = 0

    def get(self, key: K) -> V | None:
        if key not in self._data:
            self.misses += 1
            return None
        self.hits += 1
        self._data.move_to_end(key)
        return self._data[key]

    def put(self, key: K, value: V) -> None:
        if key in self._data:
            self._data.move_to_end(key)
        self._data[key] = value
        if len(self._data) > self.capacity:
            self._data.popitem(last=False)

    def keys_oldest_first(self) -> list[K]:
        return list(self._data)


def compute_delay(
    attempt: int,
    base: float,
    factor: float,
    max_delay: float,
    jitter: float,
    rand: Callable[[], float],
) -> float:
    """Delay before the retry after a zero-indexed failed `attempt`."""
    delay = min(max_delay, base * factor**attempt)
    if jitter:
        # Full jitter (jitter=1) spreads retries uniformly; partial jitter dampens.
        delay = delay * (1 - jitter) + delay * jitter * rand()
    return delay


def backoff_delays(
    attempts: int,
    base: float = 0.1,
    factor: float = 2.0,
    max_delay: float = 10.0,
    jitter: float = 0.0,
    rand: Callable[[], float] = random.random,
) -> list[float]:
    """The sleep before each retry for a plan of `attempts` total tries."""
    return [compute_delay(n, base, factor, max_delay, jitter, rand) for n in range(attempts - 1)]


def retry(
    func: Callable[[], V],
    attempts: int = 4,
    base: float = 0.1,
    factor: float = 2.0,
    max_delay: float = 10.0,
    jitter: float = 0.0,
    retry_on: tuple[type[BaseException], ...] = (Exception,),
    sleep: Callable[[float], None] = time.sleep,
    rand: Callable[[], float] = random.random,
) -> V:
    """Run `func`, retrying transient failures with capped exponential backoff."""
    if attempts < 1:
        raise ValueError("attempts must be >= 1")
    for attempt in range(attempts):
        try:
            return func()
        except retry_on:
            if attempt == attempts - 1:
                raise
            sleep(compute_delay(attempt, base, factor, max_delay, jitter, rand))
    raise RuntimeError("unreachable")


In [29]:
cache: LRUCache[str, int] = LRUCache(2)
cache.put("a", 1)
cache.put("b", 2)
assert cache.get("a") == 1          # "a" is now most recently used
cache.put("c", 3)                   # evicts "b"
assert cache.get("b") is None
assert cache.get("c") == 3
assert cache.keys_oldest_first() == ["a", "c"]
cache.put("c", 30)                  # update keeps it freshest
assert cache.get("c") == 30
cache.put("d", 4)                   # evicts "a"
assert cache.get("a") is None
print("cache hits/misses:", cache.hits, cache.misses)
assert cache.hits == 3 and cache.misses == 2

try:
    LRUCache(0)
except ValueError:
    pass
else:
    raise AssertionError("capacity 0 must raise")

# Backoff schedule is deterministic when randomness is not involved.
print("delays:", backoff_delays(4, base=0.1, factor=2.0, max_delay=10.0))
assert backoff_delays(4, base=0.1, factor=2.0, max_delay=10.0) == [0.1, 0.2, 0.4]
assert backoff_delays(5, base=1.0, factor=3.0, max_delay=5.0) == [1.0, 3.0, 5.0, 5.0]
assert backoff_delays(1) == []

calls = {"n": 0}
slept: list[float] = []


def flaky() -> str:
    calls["n"] += 1
    if calls["n"] < 3:
        raise TimeoutError("temporary")
    return "ok"


assert retry(flaky, attempts=5, base=0.1, factor=2.0, sleep=slept.append) == "ok"
assert calls["n"] == 3
print("slept between retries:", slept)
assert slept == [0.1, 0.2]          # two failures -> two sleeps


def always_fails() -> None:
    raise TimeoutError("down")


try:
    retry(always_fails, attempts=2, sleep=slept.append)
except TimeoutError:
    pass
else:
    raise AssertionError("exhausted retries must raise")

# Jitter scales the base delay by a pinned random draw.
assert compute_delay(0, base=1.0, factor=2.0, max_delay=10.0, jitter=0.5, rand=lambda: 0.0) == 0.5
assert compute_delay(0, base=1.0, factor=2.0, max_delay=10.0, jitter=0.5, rand=lambda: 1.0) == 1.0
assert retry(lambda: "ok", attempts=1) == "ok"
print("P14 checks passed")


cache hits/misses: 3 2
delays: [0.1, 0.2, 0.4]
slept between retries: [0.1, 0.2]
P14 checks passed


**Complexity & edge cases.** `OrderedDict` gives O(1) amortised `get`/`put` because it is a hash map plus a doubly linked list; `move_to_end` makes a hit the most recent and `popitem(last=False)` evicts the least recent. Edge cases: capacity zero/negative must be rejected, not silently no-op; `None` as a legal cached value is ambiguous with a miss, so use a sentinel; a hand-rolled version must update both the map and the list on every hit and eviction; retries must target idempotent, transient errors only; cap attempts and `max_delay` so a scheduler slot is not held forever; jitter is mandatory to avoid retry storms.

**Interview follow-ups.**

1. *Why does `functools.lru_cache` not solve every caching need?* It keys on hashable positional/keyword arguments, cannot expire by time, cannot be invalidated selectively, and is per-process rather than shared.
2. *What is full jitter and why does it work?* Sleep `uniform(0, min(cap, base * 2**n))`; randomising waits desynchronises clients that failed together, flattening the thundering herd when the service recovers.


## Interview questions

Twelve questions that come up when interviewers probe Python for data engineering, each with a concise model answer. Where a claim is checkable, a code cell in this section proves it.


### Q1. When do you reach for a generator instead of a list?

**Model answer.** A list materialises every result: O(n) memory and O(n) time up front. A generator produces one item per `next()` and holds O(1) state, so it can represent an unbounded or larger-than-RAM source. Use generators for file/stream ingestion and one-pass transform pipelines; use lists when you need random access, `len()`, reuse, or multiple passes.

### Q2. Why is `dict`/`set` lookup considered O(1), and what breaks it?

**Model answer.** They are hash tables: `hash(key)` picks a bucket, then equality confirms the match. Average-case lookup/insert/delete is O(1); iteration is O(n). Worst case is O(n) per operation under heavy collisions (a bad `__hash__` or adversarial keys). Keys must be hashable with value-based equality; a mutable key would break bucket invariants. Dicts preserve insertion order since Python 3.7, which is what makes an ordered dedupe a one-liner.

### Q3. `sorted()`, `list.sort()`, or `heapq.nlargest()` - which and why?

**Model answer.** `list.sort()` sorts in place, O(n log n) time, O(1) extra space (Timsort may use O(n/2) worst case), returns `None`. `sorted()` copies first, adding O(n) memory. `heapq.nlargest(k, it)` is O(n log k) time and O(k) memory, so it wins when k << n and preserves input order for ties. For top-N per group, keep one bounded heap per group.


In [30]:
import heapq
import sys

# nlargest equals a full-sort slice, but keeps memory bounded by k.
data = [5, 1, 9, 3, 7, 2]
assert heapq.nlargest(3, data) == sorted(data, reverse=True)[:3]
assert heapq.nsmallest(2, data) == sorted(data)[:2]


# A generator holds constant state regardless of how many items it could produce.
def count_up(limit):
    value = 0
    while value < limit:
        yield value
        value += 1


large_generator = count_up(10**9)
assert sys.getsizeof(large_generator) < sys.getsizeof([0] * 10_000)
assert [next(large_generator) for _ in range(3)] == [0, 1, 2]
large_generator.close()


# dict preserves insertion order, which is why ordered dedupe is easy.
pairs = [("a", 1), ("b", 2), ("a", 3)]
assert list(dict(pairs)) == ["a", "b"]     # first-seen order
assert dict(pairs)["a"] == 3               # but last write wins the value
assert list(dict.fromkeys(["x", "y", "x"])) == ["x", "y"]
print("Interview verification A passed")


Interview verification A passed


### Q4. How do you deduplicate records while preserving order, and how does it scale?

**Model answer.** Stream the data and keep a `set` of seen keys, appending the first occurrence of each: O(n) time, O(k) memory for k distinct keys. At scale, partition by key so each partition dedupes independently; or sort by key and drop adjacent duplicates (spills to disk, O(n log n)); or use a Bloom filter as a cheap pre-filter, accepting false positives. In a warehouse: `ROW_NUMBER() OVER (PARTITION BY key ORDER BY ts DESC) = 1`.

### Q5. How do you process a file larger than RAM with only the standard library?

**Model answer.** Never call `read()`/`readlines()`. Iterate the file object line by line (bounded buffer), wrap it in a generator, process in fixed batches (`itertools.batched`), and keep only aggregates or the required state. Use `gzip.open(..., "rt")` for compressed files. Parallelise by byte ranges only when the format allows seeking to a record boundary; otherwise split by file. Write aggregates out incrementally.

### Q6. `json.load` vs `json.loads` vs streaming with `raw_decode`.

**Model answer.** `json.loads(str)` parses a string; `json.load(file)` reads a file object and parses it, but both materialise the whole document. For a JSON array too large for memory, drive `json.JSONDecoder().raw_decode(buffer, index)` to pull one value at a time, or use NDJSON where every line is an independent document. Third-party options are `ijson` and `jsonlines`.


### Q7. How does an in-memory hash join work, and when does it fail?

**Model answer.** Build a hash map on the smaller (dimension) side, then stream the larger (fact) side through it: O(n + m) time, O(m) memory for the build side. It fails when the build side does not fit in memory, when one join key is badly skewed, or when null/type semantics differ (`1` vs `"1"`, NULL never equals NULL). Remedies: broadcast the small side, sort-merge join, or hash-partition by key and join partition-wise.

### Q8. How do you handle late-arriving records and duplicates?

**Model answer.** Make the sink idempotent. Keep a map keyed by the business key and retain the record with the greatest `(event_timestamp, tiebreak)`, where the tiebreak is a Kafka offset or version so replays are deterministic. If producer clocks are unreliable, prefer the ingestion timestamp for lateness. Tombstones/deletes must also win, and the downstream write must merge/upsert rather than blindly append.

### Q9. What is schema drift and how do you detect it?

**Model answer.** Schema drift is any change in columns or types between batches. Infer the observed schema (column -> set of types) per batch, compare against the catalog or the previous batch, and classify added, removed, and retyped columns. Silent type changes such as int -> string are the dangerous ones. Diff declared Parquet/Avro metadata in O(columns); added required or removed fields break readers, while int -> float is usually benign widening.


In [31]:
from collections import OrderedDict, defaultdict

# OrderedDict.move_to_end is the primitive behind LRU recency ordering.
od = OrderedDict()
for key in "abc":
    od[key] = key.upper()
od.move_to_end("a")
assert list(od) == ["b", "c", "a"]         # "a" touched most recently
od.popitem(last=False)
assert list(od) == ["c", "a"]              # oldest evicted

# A hash join with duplicate build keys multiplies rows, exactly like SQL.
build = defaultdict(list)
for key, value in [("k", 1), ("k", 2), ("x", 9)]:
    build[key].append(value)
probe = [("k", "A"), ("x", "B"), ("missing", "C")]
joined = [(key, payload, dim) for key, payload in probe for dim in build.get(key, [])]
assert joined == [("k", "A", 1), ("k", "A", 2), ("x", "B", 9)]

# Rejecting unsorted input is what keeps a binary-search partition valid.
def kth_partition_is_valid(sorted_values):
    return all(sorted_values[i] <= sorted_values[i + 1] for i in range(len(sorted_values) - 1))


assert kth_partition_is_valid([1, 2, 2, 5])
assert not kth_partition_is_valid([2, 1])
print("Interview verification B passed")


Interview verification B passed


### Q10. How do you compute percentiles and rolling windows efficiently?

**Model answer.** An exact percentile needs the sorted sample: sort once (O(n log n)) and each percentile is O(1). For rolling windows, means/sums are O(1) per element with a deque, but exact quantiles are not. Use an order-statistic tree (O(log n) insert) or an approximate sketch (t-digest, HDR histogram, Greenwald-Khanna) for large windows. Percentiles are not additive across buckets, and on a small sample p99 is just the maximum.

### Q11. How do you implement an LRU cache, and what does `functools.lru_cache` give you?

**Model answer.** Combine a hash map (O(1) lookup) with a doubly linked list or `OrderedDict` (recency order): `get` moves the key to the most-recent end, `put` inserts and evicts the least-recent end when over capacity. Both are O(1). `functools.lru_cache` is the C implementation for pure functions keyed on hashable arguments; it cannot key unhashable objects, has no time-based expiry, and caches per process. A sentinel is needed when `None` is a legal value.

### Q12. Why retry with exponential backoff and jitter?

**Model answer.** Delay before retry n is `min(cap, base * factor**n)`, plus jitter. Backoff gives a failing dependency time to recover; jitter (`uniform(0, delay)` for full jitter) desynchronises clients so they do not stampede the service the moment it returns. Retry only transient, idempotent operations (timeouts, 429, 503), cap attempts and max delay so no scheduler slot is held forever, and re-raise the final error after recording it.


In [32]:
import functools
import json

# functools.lru_cache memoizes by hashable args; cache_info exposes the counters.
calls = {"n": 0}


@functools.lru_cache(maxsize=2)
def slow_square(value):
    calls["n"] += 1
    return value * value


assert [slow_square(v) for v in (1, 2, 1, 3, 1)] == [1, 4, 1, 9, 1]
assert calls["n"] == 3                    # only 1, 2, 3 were computed
info = slow_square.cache_info()
assert (info.hits, info.misses) == (2, 3)

# raw_decode parses one value at a time out of a buffer: the streaming JSON core.
decoder = json.JSONDecoder()
text = '{"a": 1} {"b": 2}'
first, index = decoder.raw_decode(text)
second, _ = decoder.raw_decode(text, index + 1)
assert (first, second) == ({"a": 1}, {"b": 2})

# Exponential backoff is a pure function of the attempt number.
delays = [min(0.5 * 2**n, 4.0) for n in range(5)]
assert delays == [0.5, 1.0, 2.0, 4.0, 4.0]
print("Interview verification C passed")


Interview verification C passed
